# 06. CAL01 정비 감지·대기·완료와 온도 변화

**담당:** 정수진 · **작성자 표기:** jinny · **Issue:** #75 (상위 #65 → #38)  
**대상 설비:** P1-CR1-CAL01 · **수정일:** 2026-10-15  
**브랜치:** feat/75-maintenance-timeline

이번 분석은 **언제 온도가 달라졌고 → 기록된 감지까지 이어졌는가 → 착수 대기 중 어떻게 진행됐는가 → 완료 후 참조범위로 돌아왔는가**를 확인한다.
감지·착수·완료를 단순히 온도 위에 표시하는 데서 나아가 관측 범위, TC·같은 존 OP, 흔들림, 전후 차이, 과거 동일 강종 대조를 함께 남긴다.

| 먼저 볼 질문 | 확인할 근거 |
|---|---|
| 44건 중 어느 정비를 실제로 비교할 수 있나? | 단계별 관측시간·공백 표와 그림 |
| 감지 전 어떤 신호가 먼저 달라졌나? | 감지 전 참조, 지속변화 후보, 감지까지의 관측 연속성 |
| 감지·착수·완료 전후 온도와 OP는 얼마나 달라졌나? | 같은 LOT·강종의 앞뒤 15분 비교, 실제 측정값 확대 |
| 착수까지 기다리는 동안 변화가 지속됐나? | 전체 대기시간과 실제 관측시간, 동일 LOT 내 수준·IQR·변화 후보 창 |
| 완료 후 실제 회복을 관측했나? | 감지 전 참조에 대한 재진입 후보, 미관측·동시 작업 분리 |
| 그 변화가 정비 없는 구간에서도 나타나나? | 과거 동일 강종 다른 LOT 대조(정상 라벨 아님) |

**읽는 순서:** 공통 설정·원본 확인 → 새 온도 대조 절 → 기존 전체 작업표·통합 타임라인·원문 상세.
분석 코드와 해석은 이 노트북 안에 두고, 결과 이미지만 figures에 저장한다.

**시간 해석:** 대기=STRT_DT−DETC_DT, 작업=END_DT−STRT_DT. DOWN_MIN은 정지 길이이며 정확한 정지 구간이 아니다.
TBM의 감지와 온도 변화 후보를 실제 고장 발생·감지 원인으로 단정하지 않는다. 온도 관측이 없는 구간은 보간하거나 회복 0분으로 채우지 않는다.


## 공통 설정과 그림 저장 위치

아래 첫 코드 셀은 전달받은 공통 설정을 그대로 사용한다. **notebooks 폴더에서 실행**한다.
정비·부품 자료에 더해 온도 CSV의 실제 측정시각·LOT와 CR1 정기수리 캘린더를 읽는다.
REMARK의 공백·오타·빈 문자열, FAIL_MODE 빈 값, DOWN_MIN=0은 원본 그대로 보존한다.

| 폴더 | 그림과 목적 |
|---|---|
| 01_전체작업표 | 44건의 세 원시각·대기/작업/정지시간·REMARK·부품을 원문으로 확인 |
| 02_통합타임라인 | 월별 한 행 한 작업으로 정비 유형·감지→착수→완료·정기수리·온도 기록·±72시간 LOT 확인 |
| 03_시간비교 | 대기시간과 작업시간, 작업시간과 DOWN_MIN 비교 |
| 04_관심작업상세 | 주제와 WO 번호가 적힌 파일에서 실제 시각과 작업 내용을 확대 확인 |

모두 figures/06_maintenance_timeline_jinny/ 아래에 저장한다. 폴더의 README.md에도 분류와 읽는 방법을 적는다.

**색·표시 규칙:** BM=적갈색(사후정비), CBM=황갈색(상태기반정비), TBM=청회색(시간기준정비).
같은 유형은 모든 그림에서 같은 색을 쓴다. 감지=◆, 감지~착수=점선, 착수~완료=실선 막대,
착수=●, 완료=■다. 부품교체 글자는 보라색, 작업 중 온도 측정 행이 있으면 짙은 녹색,
정지시간 비교 막대는 진회색이다. CR1 정기수리 캘린더와 작업 구간이 겹치는 행의 배경은 옅은 청회색이다.
색은 정비 유형과 기록 종류를 구분하며 정상·고장·심각도를 뜻하지 않는다.


추가 폴더: **05_온도관측범위 / 06_정비별온도대조 / 07_시점확대 / 08_대조검토**.
추가 분석·시각화 코드도 이 노트북 안에서 정의하고 실행한다. 다른 파일의 분석 함수나 결과를 불러오지 않는다.


In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
ENC = "utf-8-sig"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110


In [ ]:
from pathlib import Path
import json
import math
import textwrap
import unicodedata
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown

# 공통 설정의 경로를 사용한다. 실행 위치는 notebooks 폴더다.
ROOT = Path(DATA_DIR).resolve().parent
OUTPUT_DIR = Path(FIG_DIR) / "06_maintenance_timeline_jinny"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EQP_TAG = "P1-CR1-CAL01"
COLORS = {"BM": "#B64D35", "CBM": "#B87500", "TBM": "#527A9B"}
TYPE_NAMES = {"BM": "사후정비", "CBM": "상태기반정비", "TBM": "시간기준정비"}
WAIT_COLOR, DOWN_COLOR = "#7B8794", "#3B4856"
PART_COLOR, TEMP_COLOR, PLAN_BG = "#7A5195", "#24674C", "#EDF3F7"
rcParams.update({"font.size": 11.5, "axes.titlesize": 13,
                 "axes.labelsize": 11.5, "xtick.labelsize": 10.5,
                 "ytick.labelsize": 10.5, "savefig.dpi": 170})
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
saved_figures = []
layout_issues = []
timeline_covered = []

def read_raw(path, required):
    frame = pd.read_csv(path, encoding=ENC, dtype=str, keep_default_na=False)
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{Path(path).name}: 필요한 컬럼 누락 {sorted(missing)}")
    return frame

def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for text in fig.findobj(match=lambda item: isinstance(item, plt.Text)):
        if not text.get_visible() or not text.get_text():
            continue
        box = text.get_window_extent(renderer)
        frame = fig.bbox
        if box.width > 0 and box.height > 0 and (
            box.x0 < frame.x0 - 2 or box.y0 < frame.y0 - 2
            or box.x1 > frame.x1 + 2 or box.y1 > frame.y1 + 2
        ):
            layout_issues.append((filename, text.get_text()[:80]))
    for legend in fig.legends:
        bounds = legend.get_window_extent(renderer)
        for axis in fig.axes:
            for title in [axis.title, axis._left_title, axis._right_title]:
                if title.get_text() and bounds.overlaps(title.get_window_extent(renderer)):
                    layout_issues.append((filename, "범례/제목 겹침: " + title.get_text()))
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    saved_figures.append(filename)
    plt.show()
    plt.close(fig)

def actual_clock(ax, minticks=3, maxticks=5):
    locator = mdates.AutoDateLocator(minticks=minticks, maxticks=maxticks)
    ax.xaxis.set_major_locator(locator)
    lo, hi = [mdates.num2date(value) for value in ax.get_xlim()]
    same_day = lo.date() == hi.date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M" if same_day else "%m-%d\n%H:%M"))
    ax.set_xlabel(f"원본 시각 ({lo:%Y-%m-%d})" if same_day else "2024년 원본 날짜·시각")
    ax.grid(axis="x", alpha=.18)
    ax.spines[["top", "right"]].set_visible(False)

def number(value):
    return "미기재" if pd.isna(value) else f"{value:,.0f}"

def timestamp(value):
    return "미기재/변환실패" if pd.isna(value) else value.strftime("%Y-%m-%d %H:%M")

def literal(value):
    return json.dumps(str(value), ensure_ascii=False)

def duration_text(minutes):
    if pd.isna(minutes):
        return "미기재"
    total = int(minutes)
    days, remain = divmod(total, 1440)
    hours, mins = divmod(remain, 60)
    return " ".join([part for part in [
        f"{days}일" if days else "", f"{hours}시간" if hours else "",
        f"{mins}분" if mins or not (days or hours) else ""] if part])

def wrap_units(text, limit):
    lines, line, width = [], "", 0
    for char in str(text):
        if char == "\n":
            lines.append(line)
            line, width = "", 0
            continue
        step = 2 if unicodedata.east_asian_width(char) in "WF" else 1
        if line and width + step > limit:
            split = line.rfind(" ")
            if split >= len(line) * .35:
                lines.append(line[:split + 1])
                line = line[split + 1:]
                width = sum(2 if unicodedata.east_asian_width(v) in "WF" else 1 for v in line)
            else:
                lines.append(line)
                line, width = "", 0
        line += char
        width += step
    lines.append(line)
    return "\n".join(lines)


In [ ]:
maintenance_raw = read_raw(MAINT_CSV, [
    "WO_NO", "EQP_TAG", "DETC_DT", "STRT_DT", "END_DT", "MNT_TYPE",
    "FAIL_MODE", "DOWN_MIN", "PART_CHG", "REMARK",
])
equipment_raw = read_raw(EQP_CSV, ["EQP_TAG", "EQP_NM", "EQP_TYPE"])
changes_raw = read_raw(PARTCHG_CSV, ["WO_NO", "PART_CD", "QTY", "CHG_RSN"])
part_master_raw = read_raw(PART_CSV, ["PART_CD", "PART_NM", "PART_TYPE"])

assert maintenance_raw.WO_NO.ne("").all() and maintenance_raw.WO_NO.is_unique, "정비 WO 키 확인"
assert equipment_raw.EQP_TAG.ne("").all() and equipment_raw.EQP_TAG.is_unique, "설비 키 확인"
assert part_master_raw.PART_CD.ne("").all() and part_master_raw.PART_CD.is_unique, "부품 키 확인"
equipment = equipment_raw.loc[equipment_raw.EQP_TAG.eq(EQP_TAG)]
assert len(equipment) == 1, "CAL01 설비마스터가 정확히 한 행이어야 합니다."
cal_raw = maintenance_raw.loc[maintenance_raw.EQP_TAG.eq(EQP_TAG)].copy()
assert len(cal_raw) == 44, "현재 제공본의 CAL01 44건과 다릅니다. 원본 범위를 확인하세요."

cal = cal_raw.copy()
for column in ["DETC_DT", "STRT_DT", "END_DT"]:
    cal[column + "_RAW"] = cal[column]
    cal[column] = pd.to_datetime(cal[column], format="%Y-%m-%d %H:%M", errors="coerce")
cal["DOWN_MIN_RAW"] = cal.DOWN_MIN
cal["DOWN_MIN"] = pd.to_numeric(cal.DOWN_MIN, errors="coerce")
cal["WAIT_MIN"] = (cal.STRT_DT - cal.DETC_DT).dt.total_seconds() / 60
cal["WORK_MIN"] = (cal.END_DT - cal.STRT_DT).dt.total_seconds() / 60
cal["DETECT_TO_END_MIN"] = (cal.END_DT - cal.DETC_DT).dt.total_seconds() / 60
cal["REMARK_EMPTY"] = cal.REMARK.eq("")
cal["REMARK_WHITESPACE"] = cal.REMARK.ne(cal.REMARK.str.strip())
cal = cal.sort_values(["STRT_DT", "WO_NO"], na_position="last").reset_index(drop=True)
cal["ORDER"] = np.arange(1, len(cal) + 1)

# 원행을 남긴다. 중복을 임의 삭제하거나 수량을 합산하지 않는다.
changes = changes_raw.copy()
changes["SOURCE_CSV_ROW"] = changes.index + 2
changes["EXACT_DUPLICATE"] = changes_raw.duplicated(keep=False)
cal_parts = changes.loc[changes.WO_NO.isin(cal.WO_NO)].merge(
    part_master_raw[["PART_CD", "PART_NM", "PART_TYPE"]],
    on="PART_CD", how="left", validate="many_to_one", indicator="MASTER_LINK",
)
cal_parts = cal_parts.sort_values(["WO_NO", "SOURCE_CSV_ROW"]).reset_index(drop=True)
part_counts = cal_parts.groupby("WO_NO").size()
cal["PART_ROWS"] = cal.WO_NO.map(part_counts).fillna(0).astype(int)
cal["PART_FLAG_MISMATCH"] = cal.PART_CHG.eq("Y").ne(cal.PART_ROWS.gt(0))

def parts_text(wo):
    rows = cal_parts.loc[cal_parts.WO_NO.eq(wo)]
    if rows.empty:
        return "교체 기록 없음"
    return "\n".join(
        f"{r.PART_CD} {r.PART_NM} ×{r.QTY} / {r.CHG_RSN}"
        + (" [원행 중복]" if r.EXACT_DUPLICATE else "")
        for r in rows.itertuples()
    )

cal["PARTS_DETAIL"] = cal.WO_NO.map(parts_text)
display(equipment[["EQP_TAG", "EQP_NM", "EQP_TYPE"]])
display(pd.DataFrame({
    "원본": ["정비이력 전체", "CAL01 정비", "CAL01 부품교체 원행", "CAL01 교체 WO"],
    "건수": [len(maintenance_raw), len(cal), len(cal_parts), cal_parts.WO_NO.nunique()],
}))


### 온도 측정시각·LOT·정기수리 캘린더 연결

통합 그림의 온도 기록과 가까운 LOT를 실제 CSV로 확인한다. 실제 측정 행의 시각만 사용하고 LOT 사이 공백을 채우지 않는다. CR1 정기수리는 원본 일자 범위로 표시하며 실제 정지 구간이라고 해석하지 않는다.


In [ ]:
# 온도값을 추정하거나 채우지 않는다. 실제 측정 행의 시각과 LOT만 읽는다.
temperature_times = pd.read_csv(TEMP_CSV, encoding=ENC, usecols=["MEAS_DT", "LOT_NO"],
                               dtype=str, keep_default_na=False)
temperature_times["MEAS_DT"] = pd.to_datetime(
    temperature_times.MEAS_DT, format="%Y-%m-%d %H:%M:%S", errors="raise")
assert temperature_times.LOT_NO.ne("").all() and temperature_times.MEAS_DT.notna().all()
temp_index = pd.DatetimeIndex(temperature_times.MEAS_DT.sort_values())
lot_observations = temperature_times.groupby("LOT_NO", sort=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), ROWS=("MEAS_DT", "size")
).reset_index().sort_values("START")
lot_indexes = {lot: pd.DatetimeIndex(group.MEAS_DT.sort_values())
               for lot, group in temperature_times.groupby("LOT_NO", sort=False)}

calendar_raw = read_raw(REPAIR_CSV, ["PLAN_ID", "PLANT_CD", "STRT_DT", "END_DT", "SCOPE"])
cr1_plans = calendar_raw.loc[calendar_raw.PLANT_CD.eq("CR1")].copy()
cr1_plans["START"] = pd.to_datetime(cr1_plans.STRT_DT, format="%Y-%m-%d", errors="raise")
# 일자 자료의 종료일을 포함한다. 정비 END_DT나 실제 정지 시각을 바꾸는 연산이 아니다.
cr1_plans["END_EXCL"] = pd.to_datetime(cr1_plans.END_DT, format="%Y-%m-%d", errors="raise") + pd.Timedelta(days=1)

def row_count(index, start, end, exact=False):
    if pd.isna(start) or pd.isna(end):
        return pd.NA
    return int(index.searchsorted(end, side="right" if exact else "left")
               - index.searchsorted(start, side="left"))

cal["TEMP_WORK_ROWS"] = [row_count(temp_index, r.STRT_DT, r.END_DT) for r in cal.itertuples()]
cal["TEMP_DETECT_ROWS"] = [row_count(temp_index, r.DETC_DT, r.DETC_DT, exact=True)
                           for r in cal.itertuples()]
cal["TEMP_CONTEXT"] = [
    "작업 중 기록" if r.TEMP_WORK_ROWS > 0 else
    "감지 시각 기록\n작업 중 없음" if r.TEMP_DETECT_ROWS > 0 else
    "해당 시각\n기록 없음" for r in cal.itertuples()
]
cal["PLAN_IDS"] = [
    ", ".join(cr1_plans.loc[cr1_plans.START.lt(r.END_DT) & cr1_plans.END_EXCL.gt(r.STRT_DT), "PLAN_ID"])
    for r in cal.itertuples()
]
NEARBY_HOURS = 72
def nearby_lots(detect):
    if pd.isna(detect):
        return "감지 시각 미기재"
    results = []
    for lot in lot_observations.itertuples():
        if detect < lot.START:
            distance = (lot.START - detect).total_seconds() / 3600
            label = f"{lot.LOT_NO} 시작\n{distance:.1f}시간 전 감지"
        elif detect > lot.END:
            distance = (detect - lot.END).total_seconds() / 3600
            label = f"{lot.LOT_NO} 종료\n{distance:.1f}시간 후 감지"
        else:
            distance = 0
            exists = row_count(lot_indexes[lot.LOT_NO], detect, detect, exact=True) > 0
            label = f"{lot.LOT_NO}\n감지 시각 기록" if exists else f"{lot.LOT_NO} 관측범위 안\n감지 시각 행 없음"
        if distance <= NEARBY_HOURS:
            results.append((distance, label))
    return "\n".join(label for _, label in sorted(results)) if results else "±72시간 내 없음"
cal["NEARBY_LOTS"] = cal.DETC_DT.map(nearby_lots)
display(lot_observations)
display(cr1_plans[["PLAN_ID", "STRT_DT", "END_DT", "SCOPE"]])
display(cal[["WO_NO", "TEMP_WORK_ROWS", "TEMP_DETECT_ROWS", "PLAN_IDS", "NEARBY_LOTS"]])


## 2. 시각·결측·연결 점검

시각 파싱 실패·음수 대기/작업·DOWN_MIN 결측·교체 여부와 실제 연결의 불일치를 별도로 확인한다.
FAIL_MODE와 REMARK의 빈 값은 채우지 않는다. TBM에도 감지 시각 또는 고장모드가 적힐 수 있으므로,
데이터사전의 일반 설명보다 실제 원행을 먼저 확인한다.
정비지시와 부품교체는 일대다 관계다. 작업표는 WO당 한 행을 유지하고 교체 상세는 별도 표로 남긴다.


In [ ]:
audit = pd.DataFrame([
    ("CAL01 정비", len(cal), "현재 제공본 기준 44건"),
    ("DETC_DT 미기재", int(cal.DETC_DT_RAW.eq("").sum()), "결측은 대기시간 미확인"),
    ("DETC_DT 변환 실패", int((cal.DETC_DT_RAW.ne("") & cal.DETC_DT.isna()).sum()), "원문 확인 필요"),
    ("STRT_DT 미확인", int(cal.STRT_DT.isna().sum()), "작업 시작 추정 금지"),
    ("END_DT 미확인", int(cal.END_DT.isna().sum()), "작업 종료 추정 금지"),
    ("음수 대기시간", int(cal.WAIT_MIN.lt(0).sum()), "원시각 순서 확인"),
    ("음수 작업시간", int(cal.WORK_MIN.lt(0).sum()), "원시각 순서 확인"),
    ("DOWN_MIN 미확인", int(cal.DOWN_MIN.isna().sum()), "0으로 채우지 않음"),
    ("DOWN_MIN 음수", int(cal.DOWN_MIN.lt(0).sum()), "원본 확인 필요"),
    ("DOWN_MIN = 0", int(cal.DOWN_MIN.eq(0).sum()), "기록값 0; 구체적 운전상태는 미확정"),
    ("DOWN_MIN > 작업시간", int(cal.DOWN_MIN.gt(cal.WORK_MIN).sum()), "서로 다른 시간 정의; 자동 보정 금지"),
    ("FAIL_MODE 빈 문자열", int(cal.FAIL_MODE.eq("").sum()), "고장모드로 채우지 않음"),
    ("REMARK 빈 문자열", int(cal.REMARK_EMPTY.sum()), "원문 빈 값 보존"),
    ("REMARK 앞뒤 공백", int(cal.REMARK_WHITESPACE.sum()), "원문 공백 보존"),
    ("교체 표기·연결 불일치", int(cal.PART_FLAG_MISMATCH.sum()), "PART_CHG와 WO 연결 비교"),
    ("부품마스터 미연결", int(cal_parts.MASTER_LINK.ne("both").sum()), "부품코드 그대로 표시"),
    ("CAL01 교체 원행 중복", int(cal_parts.EXACT_DUPLICATE.sum()), "원행 유지; 수량 합산하지 않음"),
], columns=["점검", "건수", "처리·해석"])
display(audit)
type_quality = cal.groupby("MNT_TYPE", dropna=False).agg(
    정비건수=("WO_NO", "size"),
    감지시각존재=("DETC_DT", "count"),
    고장모드빈값=("FAIL_MODE", lambda s: s.eq("").sum()),
    REMARK빈값=("REMARK_EMPTY", "sum"),
    부품교체WO=("PART_ROWS", lambda s: s.gt(0).sum()),
)
display(type_quality)
issues = cal.loc[
    cal.STRT_DT.isna() | cal.END_DT.isna()
    | cal.WAIT_MIN.lt(0) | cal.WORK_MIN.lt(0)
    | cal.DOWN_MIN.isna() | cal.DOWN_MIN.lt(0) | cal.PART_FLAG_MISMATCH
]
display(issues[["WO_NO", "DETC_DT_RAW", "STRT_DT_RAW", "END_DT_RAW",
                "DOWN_MIN_RAW", "PART_CHG", "PART_ROWS"]])


## 정비와 온도 대조: 먼저 비교할 수 있는 범위를 확인한다

**감지 이전 24시간 / 감지~착수 / 착수~완료 / 완료 이후 24시간**을 분리한다.
모든 44건을 유지하며, 실제로 온도 행이 있는 시간과 공백을 표시한다. 관측시간은 실제 인접 측정시각의 차이로 계산하며 실제 가동시간을 뜻하지 않는다.
강종은 가장 최근 CR1 강종변경의 AFT_VAL이 즉시 유효하다고 가정해 연결한다. 목표온도·라인속도·가동상태는 제공되지 않아 동일 운전조건이나 정상 상태를 확정하지 않는다.


#### 계산 설정과 분석 표 구조

이하 분석·시각화 코드는 모두 이 노트북 안에서 정의한다. 파일로 저장하는 결과는 figures의 이미지뿐이다.


In [ ]:
"""CAL01 maintenance review using observed sensor data, without gap filling.

The reference and thresholds below are exploratory comparison rules, not
engineering limits or validated fault labels. All stage intervals are half open.
"""

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


TC = [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
STAGES = ("감지 전", "착수 대기", "작업 중", "완료 후")
POINTS = {"DETC_DT": "감지", "STRT_DT": "착수", "END_DT": "완료"}


@dataclass(frozen=True)
class Settings:
    pre_hours: int = 24
    post_hours: int = 24
    max_gap_seconds: int = 5
    min_samples_per_minute: int = 30
    reference_minutes: int = 30
    min_reference_minutes: int = 20
    grade_settle_minutes: int = 10
    threshold: float = 4.0
    persistence_minutes: int = 3
    recovery_minutes: int = 10
    boundary_minutes: int = 15
    min_boundary_minutes: int = 3


@dataclass
class Review:
    maintenance: pd.DataFrame
    raw: pd.DataFrame
    minutes: pd.DataFrame
    events: pd.DataFrame
    coverage: pd.DataFrame
    metrics: pd.DataFrame
    boundaries: pd.DataFrame
    decisions: pd.DataFrame
    controls: pd.DataFrame
    references: pd.DataFrame
    views: dict = field(repr=False)
    settings: Settings = field(default_factory=Settings)


def windows(row, settings):
    return {
        "감지 전": (row.DETC_DT - pd.Timedelta(hours=settings.pre_hours), row.DETC_DT),
        "착수 대기": (row.DETC_DT, row.STRT_DT),
        "작업 중": (row.STRT_DT, row.END_DT),
        "완료 후": (row.END_DT, row.END_DT + pd.Timedelta(hours=settings.post_hours)),
    }


#### 원본 측정값과 1분 집계

강종 기록을 과거 방향으로 연결하고 LOT·강종·긴 공백에서 구간을 나눈다. 실제 시각을 사용하며 값을 보간하지 않는다.


In [ ]:
def prepare_temperature(temperature, events, settings):
    raw = temperature.copy().sort_values("MEAS_DT").reset_index(drop=True)
    raw["MEAS_DT"] = pd.to_datetime(raw.MEAS_DT)
    raw["LOT_NO"] = raw.LOT_NO.astype(str)
    if raw.MEAS_DT.duplicated().any():
        raise ValueError("Duplicate sensor timestamps require an explicit resolution rule.")
    grade = events.loc[
        events.PLANT_CD.eq("CR1") & events.EVT_TYPE.eq("강종변경"),
        ["EVT_DT", "AFT_VAL"],
    ].copy().sort_values("EVT_DT")
    grade["EVT_DT"] = pd.to_datetime(grade.EVT_DT)
    grade = grade.rename(columns={"EVT_DT": "GRADE_DT", "AFT_VAL": "GRADE"})
    if grade.GRADE_DT.duplicated().any():
        raise ValueError("Ambiguous simultaneous CR1 grade changes.")
    raw = pd.merge_asof(raw, grade, left_on="MEAS_DT", right_on="GRADE_DT", direction="backward")
    raw["GRADE"] = raw.GRADE.fillna("미확인")
    step = raw.MEAS_DT.diff().dt.total_seconds()
    new_segment = (step.gt(settings.max_gap_seconds) | raw.LOT_NO.ne(raw.LOT_NO.shift())
                   | raw.GRADE.ne(raw.GRADE.shift()))
    raw["SEGMENT"] = new_segment.cumsum()
    raw["MINUTE"] = raw.MEAS_DT.dt.floor("min")
    keys = ["LOT_NO", "GRADE", "SEGMENT", "MINUTE"]
    groups = raw.groupby(keys, sort=True, observed=True)
    minute = groups[TC + OP].median()
    count = groups[TC + OP].count()
    minute = minute.where(count.ge(settings.min_samples_per_minute))
    for channel in TC:
        minute[channel + "_IQR"] = (
            groups[channel].quantile(.75) - groups[channel].quantile(.25)
        ).where(count[channel].ge(settings.min_samples_per_minute))
    minute["ROWS"] = groups.size()
    minute["FIRST_DT"] = groups.MEAS_DT.min()
    minute["LAST_DT"] = groups.MEAS_DT.max()
    minute["GRADE_DT"] = groups.GRADE_DT.first()
    minute = minute.reset_index().sort_values("MINUTE").reset_index(drop=True)
    minute["AVAILABLE_AT"] = minute.MINUTE + pd.Timedelta(minutes=1)
    minute["VALID_TC"] = minute[TC].notna().all(axis=1)
    # A partial first/last minute is retained as data, but not promoted to a full
    # evaluable minute solely because its endpoints fall in the same minute.
    minute["VALID_TC"] &= (minute.LAST_DT - minute.FIRST_DT).dt.total_seconds().ge(45)
    return raw, minute


def observed_seconds(raw, start, end, max_gap_seconds=5):
    """Sum only observed adjacent intervals, clipping both stage boundaries."""
    if end <= start or len(raw) < 2:
        return 0.0
    stamp = raw.MEAS_DT.to_numpy(dtype="datetime64[ns]").astype(np.int64)
    a, b = stamp[:-1], stamp[1:]
    connected = ((raw.SEGMENT.to_numpy()[:-1] == raw.SEGMENT.to_numpy()[1:])
                 & ((b - a) <= max_gap_seconds * 1_000_000_000))
    length = np.maximum(0, np.minimum(b, pd.Timestamp(end).value)
                        - np.maximum(a, pd.Timestamp(start).value))
    return float(length[connected].sum() / 1_000_000_000)


def sustained_runs(frame, flag, minimum):
    """Runs break at a missing minute, LOT/grade/segment boundary or false flag."""
    if frame.empty:
        return []
    result, current = [], []
    for i, row in frame.iterrows():
        if not bool(flag.loc[i]):
            if len(current) >= minimum:
                result.append(current)
            current = []
            continue
        if current:
            prev = frame.loc[current[-1]]
            connected = (row.SEGMENT == prev.SEGMENT
                         and row.MINUTE - prev.MINUTE == pd.Timedelta(minutes=1))
            if not connected:
                if len(current) >= minimum:
                    result.append(current)
                current = []
        current.append(i)
    if len(current) >= minimum:
        result.append(current)
    return result


#### 감지 전 참조와 온도 변화 점수

참조는 감지 전에 완결된 동일 LOT 자료에서만 만든다. 정상 확정 구간이나 공학적 경보 한계는 아니다.


In [ ]:
def fit_reference(view, detection, settings):
    """Use only earlier observed minutes of one LOT and grade; never post data."""
    eligible = view.loc[view.VALID_TC & view.AVAILABLE_AT.le(detection)].copy()
    eligible = eligible.loc[
        eligible.GRADE.ne("미확인")
        & (eligible.MINUTE - eligible.GRADE_DT).dt.total_seconds().ge(
            settings.grade_settle_minutes * 60)
    ]
    if eligible.empty:
        return None
    # The most recent pre-detection LOT is the reference context. Other LOTs
    # during a long wait are not scored against this LOT's temperature level.
    last = eligible.iloc[-1]
    eligible = eligible.loc[eligible.LOT_NO.eq(last.LOT_NO) & eligible.GRADE.eq(last.GRADE)]
    lo = eligible.MINUTE.min()
    reference = eligible.loc[eligible.MINUTE.lt(lo + pd.Timedelta(minutes=settings.reference_minutes))]
    if len(reference) < settings.min_reference_minutes:
        return None
    center = reference[TC + OP].median()
    scale = (reference[TC + OP] - center).abs().median() * 1.4826
    floor = pd.Series({**{c: .2 for c in TC}, **{c: 1.0 for c in OP}})
    scale = scale.clip(lower=floor)
    return {"LOT_NO": last.LOT_NO, "GRADE": last.GRADE, "START": reference.MINUTE.min(),
            "END": reference.AVAILABLE_AT.max(), "N": len(reference),
            "center": center, "scale": scale}


def score_view(view, reference):
    result = view.copy()
    result["TC_SCORE"] = np.nan
    result["OP_SCORE"] = np.nan
    result["TOP_TC"] = ""
    if reference is None:
        return result
    mask = (result.LOT_NO.eq(reference["LOT_NO"]) & result.GRADE.eq(reference["GRADE"])
            & result.MINUTE.ge(reference["END"]) & result.VALID_TC)
    if mask.any():
        z = ((result.loc[mask, TC + OP] - reference["center"]) / reference["scale"]).abs()
        result.loc[mask, "TC_SCORE"] = z[TC].max(axis=1)
        result.loc[mask, "OP_SCORE"] = z[OP].max(axis=1)
        result.loc[mask, "TOP_TC"] = z[TC].idxmax(axis=1)
    return result


#### 시점 전후 차이와 착수 대기 중 추세

같은 LOT·강종의 앞뒤 15분을 비교한다. 온도 수준·1분 내 흔들림·실제 시간에 대한 기울기를 따로 계산한다.


In [ ]:
def boundary_comparisons(row, view, settings):
    records = []
    for field, label in POINTS.items():
        point = getattr(row, field)
        before = view.loc[view.MINUTE.ge(point - pd.Timedelta(minutes=settings.boundary_minutes))
                          & view.AVAILABLE_AT.le(point) & view.VALID_TC]
        after = view.loc[view.MINUTE.ge(point)
                         & view.AVAILABLE_AT.le(point + pd.Timedelta(minutes=settings.boundary_minutes))
                         & view.VALID_TC]
        common = set(zip(before.LOT_NO, before.GRADE)) & set(zip(after.LOT_NO, after.GRADE))
        usable = []
        for lot, grade in sorted(common):
            left = before.loc[before.LOT_NO.eq(lot) & before.GRADE.eq(grade)]
            right = after.loc[after.LOT_NO.eq(lot) & after.GRADE.eq(grade)]
            if min(len(left), len(right)) >= settings.min_boundary_minutes:
                usable.append((left, right, lot, grade))
        status = "비교 가능" if usable else (
            "전후 관측 없음" if before.empty and after.empty else
            "감지 전후 조건 다름" if not common and len(before) and len(after) else "한쪽 관측/분량 부족")
        for channel in TC + OP:
            rec = {"WO_NO": row.WO_NO, "POINT": label, "CHANNEL": channel, "STATUS": status,
                   "BEFORE_MINUTES": len(before), "AFTER_MINUTES": len(after),
                   "BEFORE_MEDIAN": np.nan, "AFTER_MEDIAN": np.nan, "DELTA": np.nan,
                   "BEFORE_IQR": np.nan, "AFTER_IQR": np.nan,
                   "BEFORE_SLOPE_PER_MIN": np.nan, "AFTER_SLOPE_PER_MIN": np.nan,
                   "LOT_NO": "", "GRADE": ""}
            if usable:
                left, right, lot, grade = max(usable, key=lambda item: min(len(item[0]), len(item[1])))
                rec.update(LOT_NO=lot, GRADE=grade, BEFORE_MINUTES=len(left), AFTER_MINUTES=len(right),
                           BEFORE_MEDIAN=left[channel].median(), AFTER_MEDIAN=right[channel].median())
                rec["DELTA"] = rec["AFTER_MEDIAN"] - rec["BEFORE_MEDIAN"]
                rec["BEFORE_SLOPE_PER_MIN"] = time_slope(left, channel)
                rec["AFTER_SLOPE_PER_MIN"] = time_slope(right, channel)
                if channel in TC:
                    rec.update(BEFORE_IQR=left[channel + "_IQR"].median(),
                               AFTER_IQR=right[channel + "_IQR"].median())
            records.append(rec)
    return records


def time_slope(frame, channel):
    """Descriptive trend per elapsed minute, within a continuous segment."""
    candidates = [g.loc[g[channel].notna()] for _, g in frame.groupby("SEGMENT")]
    if not candidates:
        return np.nan
    segment = max(candidates, key=len).sort_values("MINUTE")
    if len(segment) < 3 or segment.MINUTE.diff().dropna().ne(pd.Timedelta(minutes=1)).any():
        return np.nan
    elapsed = (segment.MINUTE - segment.MINUTE.min()).dt.total_seconds().to_numpy() / 60
    return float(np.polyfit(elapsed, segment[channel].to_numpy(), 1)[0])


def stage_metrics(row, view, settings):
    records = []
    for stage, (lo, hi) in windows(row, settings).items():
        part = view.loc[view.MINUTE.ge(lo) & view.MINUTE.lt(hi)]
        for (lot, grade), group in part.groupby(["LOT_NO", "GRADE"], observed=True):
            for channel in TC + OP:
                values = group.loc[group[channel].notna()]
                # The first/last ten-minute medians are within one connected
                # segment; missing calendar time is never compressed away.
                shift = np.nan
                first_level = last_level = np.nan
                eligible = values.groupby("SEGMENT").filter(lambda g: len(g) >= 20)
                if not eligible.empty:
                    segment = max((g for _, g in eligible.groupby("SEGMENT")), key=len)
                    first = segment.loc[segment.MINUTE.lt(segment.MINUTE.min() + pd.Timedelta(minutes=10))]
                    last = segment.loc[segment.MINUTE.ge(segment.MINUTE.max() - pd.Timedelta(minutes=9))]
                    if min(len(first), len(last)) >= 8:
                        first_level, last_level = first[channel].median(), last[channel].median()
                        shift = last_level - first_level
                scores = group.loc[group.TC_SCORE.notna(), "TC_SCORE"]
                records.append({"WO_NO": row.WO_NO, "STAGE": stage, "LOT_NO": lot, "GRADE": grade,
                                "CHANNEL": channel, "MINUTES": len(values),
                                "MEDIAN": values[channel].median(),
                                "MINUTE_IQR_MEDIAN": (values[channel + "_IQR"].median()
                                                       if channel in TC else np.nan),
                                "FIRST_10_MEDIAN": first_level, "LAST_10_MEDIAN": last_level,
                                "WITHIN_SEGMENT_SHIFT": shift, "SCORED_MINUTES": len(scores),
                                "TC_CHANGE_FRACTION": scores.gt(settings.threshold).mean()})
    return records


#### 지속변화·감지 선행시간·완료 후 재진입·대조

후보 시작과 지속 확인 시각을 구분한다. 공백을 건너 선행시간·회복시간을 만들어내지 않는다.


In [ ]:
def event_decision(row, view, reference, raw, maintenance, settings):
    before_raw = raw.loc[raw.MEAS_DT.lt(row.DETC_DT)
                         & raw.MEAS_DT.ge(row.DETC_DT - pd.Timedelta(hours=settings.pre_hours))]
    last_pre = before_raw.MEAS_DT.max()
    overlaps = maintenance.loc[
        maintenance.WO_NO.ne(row.WO_NO) & maintenance.STRT_DT.lt(row.END_DT)
        & maintenance.END_DT.gt(row.STRT_DT), "WO_NO"].tolist()
    result = {"WO_NO": row.WO_NO, "MNT_TYPE": row.MNT_TYPE, "REMARK": row.REMARK,
              "WAIT_MIN": (row.STRT_DT - row.DETC_DT).total_seconds() / 60,
              "WORK_MIN": (row.END_DT - row.STRT_DT).total_seconds() / 60,
              "LAST_PRE_OBSERVATION": last_pre,
              "PRE_OBSERVATION_GAP_MIN": ((row.DETC_DT - last_pre).total_seconds() / 60
                                          if pd.notna(last_pre) else np.nan),
              "REFERENCE_STATUS": "참조 있음" if reference is not None else "감지 전 참조 관측 부족",
              "FIRST_PRE_CHANGE": pd.NaT, "CHANGE_CONFIRMED_AT": pd.NaT, "CHANGE_TC": "",
              "LEAD_MIN": np.nan, "WAIT_EVALUATED_BINS": 0, "WAIT_CHANGE_BINS": np.nan,
              "LEAD_STATUS": "참조 부족", "RECOVERY_AT": pd.NaT, "RECOVERY_MIN": np.nan,
              "RECOVERY_STATUS": "감지 전 참조 부족", "OTHER_WORK_ORDERS": ", ".join(overlaps)}
    if reference is None:
        return result
    result["REFERENCE_STATUS"] = "감지 전 동일 LOT 참조(정상 미확정)"
    scored = view.loc[view.TC_SCORE.notna()]
    changed = scored.TC_SCORE.gt(settings.threshold)
    runs = sustained_runs(scored, changed, settings.persistence_minutes)
    pre = [run for run in runs if scored.loc[run[settings.persistence_minutes - 1], "AVAILABLE_AT"] <= row.DETC_DT]
    result["LEAD_STATUS"] = "현재 탐색 규칙에서 감지 전 지속변화 후보 없음"
    if pre:
        run = pre[0]
        first = scored.loc[run[0]]
        result.update(FIRST_PRE_CHANGE=first.MINUTE, CHANGE_TC=first.TOP_TC,
                      CHANGE_CONFIRMED_AT=scored.loc[run[settings.persistence_minutes - 1], "AVAILABLE_AT"])
        # A numerical lead is allowed only if that same sustained run reaches
        # the detection boundary. Old transients across an observation gap
        # are not presented as a fault detection lead time.
        continuing = [r for r in pre if scored.loc[r[-1], "AVAILABLE_AT"] >= row.DETC_DT]
        if continuing and result["PRE_OBSERVATION_GAP_MIN"] <= 2:
            run = continuing[-1]
            result.update(FIRST_PRE_CHANGE=scored.loc[run[0], "MINUTE"],
                          CHANGE_CONFIRMED_AT=scored.loc[run[settings.persistence_minutes - 1], "AVAILABLE_AT"],
                          CHANGE_TC=scored.loc[run[0], "TOP_TC"],
                          LEAD_MIN=(row.DETC_DT - scored.loc[run[0], "MINUTE"]).total_seconds() / 60,
                          LEAD_STATUS="감지까지 이어진 변화 후보(원인 미확정)")
        else:
            result["LEAD_STATUS"] = "이전 변화 후보 있음; 감지까지 연속성 미확인"
    wait = scored.loc[scored.MINUTE.ge(row.DETC_DT) & scored.MINUTE.lt(row.STRT_DT)]
    result.update(WAIT_EVALUATED_BINS=len(wait),
                  WAIT_CHANGE_BINS=int(wait.PERSISTENT_CHANGE.sum()) if len(wait) else np.nan)
    after = scored.loc[scored.MINUTE.ge(row.END_DT)]
    result["RECOVERY_STATUS"] = "완료 후 동일 LOT 평가 관측 없음"
    if after.empty:
        return result
    prior = scored.loc[scored.MINUTE.lt(row.END_DT) & scored.MINUTE.ge(row.DETC_DT)]
    prior_runs = sustained_runs(prior, prior.TC_SCORE.gt(settings.threshold), settings.persistence_minutes)
    if not prior_runs:
        result["RECOVERY_STATUS"] = "작업 전·중 지속변화 미확인; 회복 판정 대상 아님"
        return result
    stable = sustained_runs(after, after.TC_SCORE.le(settings.threshold), settings.recovery_minutes)
    if not stable:
        result["RECOVERY_STATUS"] = "완료 후 참조범위 지속 재진입 미관측"
        return result
    first = after.loc[stable[0][0], "MINUTE"]
    # If there is a gap before the first stable minute, the recovery may have
    # occurred inside that gap: report a bound, never an exact recovery time.
    duration = (first - row.END_DT).total_seconds()
    evaluation_end = first + pd.Timedelta(minutes=1)
    coverage = observed_seconds(raw, row.END_DT, evaluation_end, settings.max_gap_seconds)
    completion_observed = observed_seconds(
        raw, row.END_DT, row.END_DT + pd.Timedelta(seconds=2), settings.max_gap_seconds) >= 1
    if not completion_observed or coverage < (duration + 60) * .9:
        result["RECOVERY_STATUS"] = "공백 뒤 참조범위 재진입 관측; 정확한 회복시간 미확인"
        return result
    result.update(RECOVERY_AT=first, RECOVERY_MIN=duration / 60,
                  RECOVERY_STATUS="참조범위 재진입 후보" + ("; 동시 작업으로 효과 분리 불가" if overlaps else ""))
    return result


def control_comparison(row, view, reference, minutes, maintenance, settings):
    rec = {"WO_NO": row.WO_NO, "GRADE": "", "EVENT_MINUTES": 0,
           "EVENT_CHANGE_FRACTION": np.nan, "CONTROL_MINUTES": 0,
           "CONTROL_CHANGE_FRACTION": np.nan, "CONTROL_LOTS": "",
           "STATUS": "감지 전 참조 부족"}
    if reference is None:
        return rec
    rec["GRADE"] = reference["GRADE"]
    event = view.loc[view.TC_SCORE.notna()]
    rec.update(EVENT_MINUTES=len(event), EVENT_CHANGE_FRACTION=event.TC_SCORE.gt(settings.threshold).mean())
    eligible = minutes.loc[
        minutes.GRADE.eq(reference["GRADE"]) & minutes.LOT_NO.ne(reference["LOT_NO"])
        & minutes.AVAILABLE_AT.le(reference["START"]) & minutes.VALID_TC
    ].copy()
    for other in maintenance.itertuples():
        lo = other.DETC_DT - pd.Timedelta(hours=settings.pre_hours)
        hi = other.END_DT + pd.Timedelta(hours=settings.post_hours)
        eligible = eligible.loc[eligible.AVAILABLE_AT.le(lo) | eligible.MINUTE.ge(hi)]
    if len(eligible) < settings.min_reference_minutes:
        rec["STATUS"] = "과거 동일 강종 대조 관측 부족"
        return rec
    z = ((eligible[TC] - reference["center"][TC]) / reference["scale"][TC]).abs().max(axis=1)
    rec.update(CONTROL_MINUTES=len(eligible), CONTROL_CHANGE_FRACTION=z.gt(settings.threshold).mean(),
               CONTROL_LOTS=", ".join(eligible.LOT_NO.unique()),
               STATUS="과거 동일 강종 대조(정상·동일 운전조건 미확정)")
    return rec


#### 44건 분석 실행 함수와 검증

미관측, 참조 부족, 조건 불일치를 NA로 보존하고 44건의 모든 단계를 유지한다.


In [ ]:
def analyze(maintenance, temperature, events, settings=None):
    settings = settings or Settings()
    maintenance = maintenance.copy()
    for field in POINTS:
        maintenance[field] = pd.to_datetime(maintenance[field])
    maintenance = maintenance.loc[maintenance.EQP_TAG.eq("P1-CR1-CAL01")].sort_values("DETC_DT")
    if not maintenance.WO_NO.is_unique:
        raise ValueError("WO_NO must be unique.")
    if ((maintenance.STRT_DT < maintenance.DETC_DT) | (maintenance.END_DT < maintenance.STRT_DT)).any():
        raise ValueError("Invalid maintenance time order.")
    events = events.copy()
    events["EVT_DT"] = pd.to_datetime(events.EVT_DT)
    raw, minutes = prepare_temperature(temperature, events, settings)
    coverage, metrics, boundaries, decisions, controls, refs, views = [], [], [], [], [], [], {}
    for row in maintenance.itertuples():
        stage_windows = windows(row, settings)
        lo, hi = stage_windows["감지 전"][0], stage_windows["완료 후"][1]
        view = minutes.loc[minutes.MINUTE.ge(lo) & minutes.MINUTE.lt(hi)].copy()
        reference = fit_reference(view, row.DETC_DT, settings)
        view = score_view(view, reference)
        view["PERSISTENT_CHANGE"] = False
        for run in sustained_runs(view, view.TC_SCORE.gt(settings.threshold), settings.persistence_minutes):
            view.loc[run, "PERSISTENT_CHANGE"] = True
        views[row.WO_NO] = view
        if reference is not None:
            refs.append({"WO_NO": row.WO_NO, **{k: v for k, v in reference.items() if k not in ("center", "scale")}})
        for stage, (a, b) in stage_windows.items():
            observed = raw.loc[raw.MEAS_DT.ge(a) & raw.MEAS_DT.lt(b)]
            seconds = observed_seconds(raw, a, b, settings.max_gap_seconds)
            total = (b - a).total_seconds()
            coverage.append({"WO_NO": row.WO_NO, "STAGE": stage, "START": a, "END": b,
                             "WINDOW_MIN": total / 60, "ROWS": len(observed),
                             "OBSERVED_MIN": seconds / 60,
                             "COVERAGE": seconds / total if total else np.nan,
                             "FIRST_DT": observed.MEAS_DT.min(), "LAST_DT": observed.MEAS_DT.max(),
                             "LOTS": ", ".join(observed.LOT_NO.unique()),
                             "GRADES": ", ".join(observed.GRADE.unique())})
        metrics.extend(stage_metrics(row, view, settings))
        boundaries.extend(boundary_comparisons(row, view, settings))
        decisions.append(event_decision(row, view, reference, raw, maintenance, settings))
        controls.append(control_comparison(row, view, reference, minutes, maintenance, settings))
    result = Review(maintenance, raw, minutes, events, pd.DataFrame(coverage), pd.DataFrame(metrics),
                    pd.DataFrame(boundaries), pd.DataFrame(decisions), pd.DataFrame(controls),
                    pd.DataFrame(refs), views, settings)
    validate_review(result)
    return result


def validate_review(review):
    coverage = review.coverage
    assert len(coverage) == len(review.maintenance) * len(STAGES)
    assert not coverage.duplicated(["WO_NO", "STAGE"]).any()
    assert coverage.OBSERVED_MIN.ge(0).all()
    assert coverage.OBSERVED_MIN.le(coverage.WINDOW_MIN + 1e-8).all()
    no_data = coverage.ROWS.eq(0)
    assert coverage.loc[no_data, "OBSERVED_MIN"].eq(0).all()
    assert len(review.decisions) == len(review.maintenance)
    if not review.references.empty:
        linked = review.references.merge(review.maintenance[["WO_NO", "DETC_DT"]], on="WO_NO", validate="one_to_one")
        assert linked.END.le(linked.DETC_DT).all(), "Reference used future data."
    missing_reference = review.decisions.REFERENCE_STATUS.eq("감지 전 참조 관측 부족")
    assert review.decisions.loc[missing_reference, ["LEAD_MIN", "RECOVERY_MIN"]].isna().all().all()


def load_review(data_dir, settings=None):
    data_dir = Path(data_dir)
    maintenance = pd.read_csv(data_dir / "G-02_정비이력.csv", encoding="utf-8-sig", keep_default_na=False)
    temperature = pd.read_csv(data_dir / "T-CR1-CAL01_온도.csv", encoding="utf-8-sig")
    events = pd.read_csv(data_dir / "G-02_조업이벤트.csv", encoding="utf-8-sig", keep_default_na=False)
    return analyze(maintenance, temperature, events, settings)


#### 관측범위와 전후 TC·OP 차이 그림 함수


In [ ]:
"""Static review figures and reproducible report for maintenance_temperature."""

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib import colors, font_manager, rcParams
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd



EVENT_COLORS = {"DETC_DT": "#b34e25", "STRT_DT": "#74419b", "END_DT": "#237658"}
SIGNAL_COLORS = ["#186d9b", "#db7041", "#288467", "#9a579d", "#b69027"]


def configure_fonts():
    installed = {font.name for font in font_manager.fontManager.ttflist}
    for family in ("Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"):
        if family in installed:
            rcParams["font.family"] = family
            break
    rcParams["axes.unicode_minus"] = False


def save(fig, output, relative, paths):
    path = output / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=155, bbox_inches="tight", facecolor="white")
    paths.append(path)
    plt.close(fig)


def clock(ax):
    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.set_xlabel("2024년 실제 관측 날짜·시각")


def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=.17)
    ax.tick_params(labelsize=9)


def fmt(value, digits=1):
    return "미관측/계산 불가" if pd.isna(value) else f"{value:,.{digits}f}"


def coverage_figures(review, output, paths):
    table = review.coverage.set_index(["WO_NO", "STAGE"])
    tasks = list(review.maintenance.itertuples())
    for page, start in enumerate(range(0, len(tasks), 22), 1):
        subset = tasks[start:start + 22]
        fig, axes = plt.subplots(1, 2, figsize=(16, 2.6 + len(subset) * .37),
                                 gridspec_kw={"width_ratios": [3.5, 1.45]})
        fig.subplots_adjust(left=.20, right=.96, top=.85, bottom=.12, wspace=.24)
        fig.suptitle(f"정비별로 어디까지 온도를 확인할 수 있는가?  ({page}/2)",
                     x=.025, y=.98, ha="left", fontsize=17, weight="bold")
        fig.text(.025, .93, "숫자 = 실제 인접 관측시간 / 해당 단계 전체시간의 비율  |  회색 = 관측 없음",
                 fontsize=10, color="#405465")
        cmap = plt.get_cmap("Blues").copy()
        cmap.set_bad("#e9edf1")
        matrix = np.array([[table.loc[(r.WO_NO, s), "COVERAGE"] for s in STAGES] for r in subset])
        masked = np.ma.masked_where((matrix <= 0) | np.isnan(matrix), matrix)
        axes[0].imshow(masked, aspect="auto", cmap=cmap, norm=colors.PowerNorm(.35, vmin=0, vmax=1))
        axes[0].set_xticks(range(4), ["감지 전 24시간", "감지 → 착수", "착수 → 완료", "완료 후 24시간"])
        axes[0].xaxis.tick_top()
        axes[0].set_yticks(range(len(subset)), [f"{r.WO_NO}  {r.MNT_TYPE}\n감지 {r.DETC_DT:%m-%d %H:%M}" for r in subset], fontsize=9)
        for i, row in enumerate(subset):
            for j, stage in enumerate(STAGES):
                rec = table.loc[(row.WO_NO, stage)]
                text = (f"{rec.OBSERVED_MIN:.1f}분 · {rec.COVERAGE:.1%}" if rec.ROWS else "관측 없음")
                axes[0].text(j, i, text, ha="center", va="center", fontsize=8.7,
                             color="white" if rec.COVERAGE > .3 else "#30485a")
        axes[0].set_xticks(np.arange(-.5, 4, 1), minor=True)
        axes[0].set_yticks(np.arange(-.5, len(subset), 1), minor=True)
        axes[0].grid(which="minor", color="white", linewidth=2)
        axes[0].tick_params(which="minor", left=False, bottom=False)
        wait = [(r.STRT_DT - r.DETC_DT).total_seconds() / 60 for r in subset]
        work = [(r.END_DT - r.STRT_DT).total_seconds() / 60 for r in subset]
        y = np.arange(len(subset))
        axes[1].barh(y - .16, wait, height=.29, color="#b68a3b", label="대기")
        axes[1].barh(y + .16, work, height=.29, color="#816499", label="작업")
        axes[1].set_ylim(len(subset) - .5, -.5)
        axes[1].set_yticks([])
        axes[1].set_xscale("symlog", linthresh=60)
        axes[1].set_xlabel("기록된 시간 (분; 60분 이후 로그)", fontsize=9)
        axes[1].set_title("대기와 작업의 전체 길이", fontsize=11, pad=12)
        axes[1].legend(frameon=False, ncol=2, fontsize=9, loc="lower right")
        style(axes[1])
        fig.text(.025, .045, "관측시간은 같은 LOT·강종에서 5초 이내인 인접 시각 간격의 합이다. 공백을 채우지 않으며 가동시간·정지시간을 뜻하지 않는다.", fontsize=9)
        save(fig, output, f"05_온도관측범위/{page:02d}_44건_단계별_관측범위.png", paths)


def boundary_figure(review, output, paths):
    source = review.boundaries.loc[review.boundaries.STATUS.eq("비교 가능")].copy()
    if source.empty:
        return
    combos = source[["WO_NO", "POINT", "BEFORE_MINUTES", "AFTER_MINUTES", "GRADE"]].drop_duplicates()
    order = {"감지": 0, "착수": 1, "완료": 2}
    combos["ORDER"] = combos.POINT.map(order)
    combos = combos.sort_values(["WO_NO", "ORDER"])
    fig, axes = plt.subplots(1, 2, figsize=(17, 10), gridspec_kw={"width_ratios": [1.25, 1]})
    fig.subplots_adjust(left=.07, right=.95, bottom=.24, top=.78, wspace=.30)
    fig.suptitle("감지·착수·완료 전후에 어느 신호가 얼마나 달라졌는가?", x=.025, y=.98,
                 ha="left", fontsize=18, weight="bold")
    fig.text(.025, .93, "각 시점 앞뒤 15분의 1분 중앙값 비교 · 같은 LOT·강종만 연결 · TC와 같은 존 OP를 함께 확인", fontsize=10)
    for ax, channels, title in zip(axes, [TC, OP], ["온도: 이후 - 이전 (℃ 표시 가정)", "제어출력: 이후 - 이전 (%p 표시 가정)"]):
        values = np.array([[source.loc[source.WO_NO.eq(r.WO_NO) & source.POINT.eq(r.POINT)
                                      & source.CHANNEL.eq(c), "DELTA"].iloc[0]
                            for r in combos.itertuples()] for c in channels])
        scale = max(float(np.nanmax(np.abs(values))), .1)
        im = ax.imshow(values, aspect="auto", cmap="RdBu_r", vmin=-scale, vmax=scale)
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                ax.text(j, i, f"{values[i, j]:+.1f}", ha="center", va="center", fontsize=9,
                        color="white" if abs(values[i, j]) > scale * .55 else "#263c4c")
        ax.set_yticks(range(len(channels)), channels)
        ax.set_xticks(range(len(combos)), [f"{r.WO_NO[-5:]} {r.POINT}\n전 {r.BEFORE_MINUTES} / 후 {r.AFTER_MINUTES}분\n{r.GRADE}" for r in combos.itertuples()], rotation=35, ha="right", fontsize=9)
        ax.set_title(title, loc="left", fontsize=12, pad=12)
        fig.colorbar(im, ax=ax, fraction=.025, pad=.03)
    fig.text(.025, .105, "01859 감지: 이전 관측은 5분뿐이고 관측 시작과 강종변경이 겹친다. 01854와 01859는 실제 작업 구간도 겹친다.", fontsize=10, color="#5c4a35")
    fig.text(.025, .065, "온도차는 고장 원인·정비 효과가 아니다. 특히 출측·냉각 온도의 주기적 변화와 강종변경 영향을 별도로 확인해야 한다.", fontsize=10)
    save(fig, output, "07_시점확대/00_감지_착수_완료_온도_OP_전후차이.png", paths)


#### 실제 관측 구간·정비 시각·참조 점수 표시 함수


In [ ]:
def observation_spans(raw, lo, hi):
    selected = raw.loc[raw.MEAS_DT.ge(lo) & raw.MEAS_DT.le(hi)]
    return [(g.MEAS_DT.min(), g.MEAS_DT.max()) for _, g in selected.groupby("SEGMENT")]


def context(axes, review, row, lo, hi):
    spans = observation_spans(review.raw, lo, hi)
    for ax in axes:
        ax.set_xlim(lo, hi)
        style(ax)
        cursor = lo
        for a, b in spans:
            if a > cursor:
                ax.axvspan(cursor, a, color="#e9edf1", alpha=.8, linewidth=0, zorder=-3)
            cursor = max(cursor, b)
        if cursor < hi:
            ax.axvspan(cursor, hi, color="#e9edf1", alpha=.8, linewidth=0, zorder=-3)
        if row.STRT_DT > row.DETC_DT:
            if row.DETC_DT < hi and row.STRT_DT > lo:
                ax.axvspan(max(lo, row.DETC_DT), min(hi, row.STRT_DT), color="#e5bb6c", alpha=.13, zorder=-2)
        if row.END_DT > lo and row.STRT_DT < hi:
            ax.axvspan(max(lo, row.STRT_DT), min(hi, row.END_DT), color="#a988bf", alpha=.13, zorder=-2)
        for field, label in POINTS.items():
            when = getattr(row, field)
            if lo <= when <= hi:
                ax.axvline(when, color=EVENT_COLORS[field], ls="--", lw=1.1, zorder=2)
        events = review.events.loc[review.events.PLANT_CD.eq("CR1")
                                   & review.events.EVT_DT.between(lo, hi)]
        for event in events.itertuples():
            ax.axvline(event.EVT_DT, color="#596f84", ls=":", alpha=.65, lw=1)
    return spans


def maintenance_strip(ax, review, row, lo, hi):
    others = review.maintenance.loc[review.maintenance.WO_NO.ne(row.WO_NO)
                                    & review.maintenance.STRT_DT.lt(hi)
                                    & review.maintenance.END_DT.gt(lo)]
    rows = [row] + list(others.itertuples())
    for i, task in enumerate(rows):
        ax.plot([task.DETC_DT, task.STRT_DT], [i, i], ls=":", lw=2, color="#ae8439", clip_on=True)
        ax.plot([task.STRT_DT, task.END_DT], [i, i], lw=7, color="#87619f", solid_capstyle="butt")
        for field, marker in [("DETC_DT", "D"), ("STRT_DT", "o"), ("END_DT", "s")]:
            when = getattr(task, field)
            if lo <= when <= hi:
                ax.scatter([when], [i], s=36, marker=marker, color=EVENT_COLORS[field], zorder=4)
        ax.text(.005, i + .20, f"{task.WO_NO} · {task.MNT_TYPE}", transform=ax.get_yaxis_transform(), fontsize=8, va="bottom")
    ax.set_ylim(len(rows) - .5, -.6)
    ax.set_yticks([])
    ax.set_ylabel("정비 기록", fontsize=9)
    ax.set_xlim(lo, hi)


def plot_channels(ax, frame, channels, raw=False):
    clock_col = "MEAS_DT" if raw else "MINUTE"
    for j, channel in enumerate(channels):
        for k, (_, group) in enumerate(frame.groupby("SEGMENT", sort=False)):
            ax.plot(group[clock_col], group[channel], color=SIGNAL_COLORS[j % len(SIGNAL_COLORS)],
                    lw=.8 if raw else 1.1, label=channel if k == 0 else None)
    ax.legend(frameon=False, ncol=min(5, len(channels)), fontsize=8.5, loc="upper left")


def plot_score(ax, review, row, view):
    valid = view.loc[view.TC_SCORE.notna()]
    refs = review.references.loc[review.references.WO_NO.eq(row.WO_NO)] if len(review.references) else pd.DataFrame()
    if not refs.empty:
        ref = refs.iloc[0]
        ax.axvspan(ref.START, ref.END, color="#8fb0c6", alpha=.22)
        for _, group in valid.groupby("SEGMENT"):
            ax.plot(group.MINUTE, group.TC_SCORE, color="#a3453e", lw=1)
        persistent = valid.loc[valid.PERSISTENT_CHANGE]
        ax.scatter(persistent.MINUTE, persistent.TC_SCORE, s=7, color="#a3453e", alpha=.6)
        ax.axhline(review.settings.threshold, color="#a3453e", ls=":", lw=1)
        if len(valid):
            ax.set_yscale("symlog", linthresh=review.settings.threshold)
            ax.set_ylim(bottom=0)
        ax.text(.01, .93, "청색: 감지 전 참조 · 점선 4 초과 3분 지속 시 붉은 점으로 표시",
                transform=ax.transAxes, va="top", fontsize=8.5)
    else:
        ax.text(.02, .48, "감지 전 동일 LOT 참조가 부족해 변화 점수·선행시간·회복시간을 계산하지 않음",
                transform=ax.transAxes, fontsize=9, va="center", color="#607181")
        ax.set_yticks([])
    ax.set_ylabel("TC 11개 최대\n참조 이탈 점수", fontsize=9)


#### 정비 전체 시간축과 감지·착수·완료 확대 함수


In [ ]:
def event_figure(review, row, output, paths):
    view = review.views[row.WO_NO]
    lo = row.DETC_DT - pd.Timedelta(hours=review.settings.pre_hours)
    hi = row.END_DT + pd.Timedelta(hours=review.settings.post_hours)
    decision = review.decisions.set_index("WO_NO").loc[row.WO_NO]
    fig, axes = plt.subplots(6, 1, figsize=(17, 13), sharex=True,
                             gridspec_kw={"height_ratios": [1, 1.2, 1.5, 1.5, 1.4, 1.15]})
    fig.subplots_adjust(left=.075, right=.97, top=.84, bottom=.16, hspace=.16)
    fig.suptitle(f"{row.WO_NO} · {row.MNT_TYPE} · 감지 전 → 대기 → 작업 → 완료 후", x=.025, y=.985,
                 ha="left", fontsize=18, weight="bold")
    fig.text(.025, .951, f"감지 {row.DETC_DT:%m-%d %H:%M}   착수 {row.STRT_DT:%m-%d %H:%M}   완료 {row.END_DT:%m-%d %H:%M}   |   대기 {decision.WAIT_MIN:,.0f}분 · 작업 {decision.WORK_MIN:,.0f}분", fontsize=11)
    fig.text(.025, .921, f"REMARK 원문: {row.REMARK!r}   |   FAIL_MODE: {row.FAIL_MODE!r}", fontsize=10)
    handles = [Line2D([], [], ls="--", color=EVENT_COLORS[k], label=v) for k, v in POINTS.items()]
    handles += [Patch(color="#e9edf1", label="온도 관측 없음"), Patch(color="#e5bb6c", alpha=.3, label="착수 대기"),
                Patch(color="#a988bf", alpha=.3, label="작업 구간"), Line2D([], [], color="#596f84", ls=":", label="CR1 강종 이벤트")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.02, .901), frameon=False, ncol=7, fontsize=9)
    context(axes, review, row, lo, hi)
    maintenance_strip(axes[0], review, row, lo, hi)
    plot_channels(axes[1], view, ["TC-Z1", "TC-Z2"])
    axes[1].set_ylabel("TC-Z1·2\n온도 (℃ 가정)", fontsize=9)
    plot_channels(axes[2], view, [f"TC-Z{i}" for i in range(3, 7)])
    axes[2].set_ylabel("TC-Z3~6\n온도 (℃ 가정)", fontsize=9)
    plot_channels(axes[3], view, ["TC-Z7", "TC-Z8", "TC-Z9", "TC-OUT1", "TC-OUT2"])
    axes[3].set_ylabel("후단·출측 TC\n온도 (℃ 가정)", fontsize=9)
    plot_channels(axes[4], view, ["OP-Z3", "OP-Z4", "OP-Z7", "OP-Z8"])
    axes[4].set_ylabel("제어출력 OP\n(% 표시 가정)", fontsize=9)
    plot_score(axes[5], review, row, view)
    clock(axes[-1])
    for ax in axes[1:5]:
        if view.empty:
            ax.text(.5, .5, "이 구간에 온도 관측 없음", transform=ax.transAxes, ha="center")
    first_change = (f"{decision.FIRST_PRE_CHANGE:%m-%d %H:%M} / {decision.CHANGE_TC}"
                    if pd.notna(decision.FIRST_PRE_CHANGE) else "현재 규칙에서 확인 불가/없음")
    note = (f"감지 직전 관측과의 간격 {fmt(decision.PRE_OBSERVATION_GAP_MIN)}분 · 이전 변화 후보 {first_change}\n"
            f"{decision.LEAD_STATUS} · 완료 후: {decision.RECOVERY_STATUS}\n"
            f"대기: 평가 가능한 1분창 {decision.WAIT_EVALUATED_BINS}개, 변화 후보 창 {fmt(decision.WAIT_CHANGE_BINS, 0)}개\n"
            f"동시 작업: {decision.OTHER_WORK_ORDERS or '없음'} · 회색 공백은 연결·보간하지 않음 · 강종은 최근 CR1 변경 기록을 연결한 가정")
    fig.text(.025, .035, note, fontsize=8.5, color="#445b6b", linespacing=1.6)
    save(fig, output, f"06_정비별온도대조/{row.WO_NO}_전체.png", paths)


def zoom_figure(review, row, field, output, paths):
    point, label = getattr(row, field), POINTS[field]
    lo, hi = point - pd.Timedelta(minutes=30), point + pd.Timedelta(minutes=45)
    raw = review.raw.loc[review.raw.MEAS_DT.between(lo, hi)].copy()
    if raw.empty:
        return
    view = review.views[row.WO_NO]
    view = view.loc[view.MINUTE.between(lo, hi)]
    boundary = review.boundaries.loc[review.boundaries.WO_NO.eq(row.WO_NO)
                                     & review.boundaries.POINT.eq(label)]
    usable = boundary.loc[boundary.CHANNEL.isin(TC) & boundary.DELTA.notna()]
    # Explicit descriptive selection: largest absolute before/after TC median
    # difference; when paired comparison is unavailable, show TC-Z3.
    channel = (usable.loc[usable.DELTA.abs().idxmax(), "CHANNEL"] if len(usable) else "TC-Z3")
    paired = channel.replace("TC-", "OP-")
    op_channels = [paired] if paired in OP else ["OP-Z3", "OP-Z4"]
    fig, axes = plt.subplots(5, 1, figsize=(16, 11.5), sharex=True,
                             gridspec_kw={"height_ratios": [.85, 1.8, 1.5, 1.15, 1.1]})
    fig.subplots_adjust(left=.085, right=.97, top=.81, bottom=.15, hspace=.19)
    fig.suptitle(f"{row.WO_NO} · {label} {point:%m-%d %H:%M} 주변 확대", x=.025, y=.985,
                 ha="left", fontsize=18, weight="bold")
    fig.text(.025, .945, f"{row.MNT_TYPE} · REMARK {row.REMARK!r} · 원본 1~수초 측정값과 1분 집계를 함께 확인", fontsize=10.5)
    handles = [Line2D([], [], ls="--", color=EVENT_COLORS[k], label=v) for k, v in POINTS.items()]
    handles += [Patch(color="#e9edf1", label="온도 관측 없음"), Line2D([], [], color="#596f84", ls=":", label="강종 변경")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.02, .917), frameon=False, ncol=5, fontsize=9)
    context(axes, review, row, lo, hi)
    maintenance_strip(axes[0], review, row, lo, hi)
    plot_channels(axes[1], raw, [channel], raw=True)
    axes[1].set_ylabel(f"{channel}\n온도 (℃ 가정)", fontsize=10)
    plot_channels(axes[2], raw, op_channels, raw=True)
    axes[2].set_ylabel("같은 존 OP\n(% 가정)" if paired in OP else "보조 OP-Z3·4 (% 가정)\n해당 TC의 OP 없음", fontsize=9)
    for _, group in view.groupby("SEGMENT"):
        axes[3].plot(group.MINUTE, group[channel + "_IQR"], color="#bd7047", lw=1)
    axes[3].set_ylabel("1분 내 TC IQR\n흔들림 (℃ 가정)", fontsize=9)
    plot_score(axes[4], review, row, view)
    clock(axes[-1])
    rec = boundary.loc[boundary.CHANNEL.eq(channel)].iloc[0]
    if pd.notna(rec.DELTA):
        note = (f"앞뒤 15분 중 실제 평가 창: 전 {rec.BEFORE_MINUTES} / 후 {rec.AFTER_MINUTES}분 · {rec.GRADE} / LOT {rec.LOT_NO}\n"
                f"{channel}: 중앙값 {rec.BEFORE_MEDIAN:.3f} → {rec.AFTER_MEDIAN:.3f} (차이 {rec.DELTA:+.3f}) · "
                f"1분 내 IQR 중앙값 {rec.BEFORE_IQR:.3f} → {rec.AFTER_IQR:.3f}\n"
                f"분당 온도 추세: 전 {rec.BEFORE_SLOPE_PER_MIN:+.3f} → 후 {rec.AFTER_SLOPE_PER_MIN:+.3f} (연속 구간 회귀 기울기)")
    else:
        note = f"앞뒤 15분의 동일 LOT 비교: {rec.STATUS} · 전 {rec.BEFORE_MINUTES} / 후 {rec.AFTER_MINUTES}분\n관측 종료·시작을 실제 정지·회복 시각으로 해석하지 않음"
    fig.text(.025, .055, note, fontsize=9.5, color="#3b5263", linespacing=1.6)
    fig.text(.025, .025, "TC 선택: 전후 중앙값 차이가 가장 큰 채널(전후 비교가 불가능하면 TC-Z3). 차이는 사후 관찰이며 감지 원인·정비 효과를 확정하지 않는다.", fontsize=9)
    save(fig, output, f"07_시점확대/{row.WO_NO}_{label}_확대.png", paths)


#### 과거 동일 강종 대조 그림 함수


In [ ]:
def controls_figure(review, output, paths):
    data = review.controls.loc[review.controls.EVENT_MINUTES.gt(0)].copy()
    if data.empty:
        return
    fig, ax = plt.subplots(figsize=(13, 7))
    fig.subplots_adjust(left=.28, right=.95, top=.78, bottom=.23)
    fig.suptitle("이 온도 변화가 정비 주변에서만 나타나는가?", x=.025, y=.97,
                 ha="left", fontsize=18, weight="bold")
    fig.text(.025, .91, "동일한 참조·임곗값을 과거 동일 강종의 다른 LOT에도 적용 · 다른 정비의 감지 전~완료 후 24시간은 대조에서 제외", fontsize=10)
    y = np.arange(len(data))
    ax.barh(y - .16, data.EVENT_CHANGE_FRACTION * 100, height=.28, color="#ad6f49", label="해당 WO 주변 평가 창")
    ax.barh(y + .16, data.CONTROL_CHANGE_FRACTION * 100, height=.28, color="#4f8aa9", label="과거 동일 강종 대조 창")
    for i, rec in enumerate(data.itertuples()):
        if pd.isna(rec.CONTROL_CHANGE_FRACTION):
            ax.text(2, i + .18, "대조 관측 부족", va="center", fontsize=9, color="#667887")
    ax.set_yticks(y, [f"{r.WO_NO} · {r.GRADE}\n평가 {r.EVENT_MINUTES} / 대조 {r.CONTROL_MINUTES}분창" for r in data.itertuples()], fontsize=10)
    ax.set_xlim(0, 105)
    ax.invert_yaxis()
    ax.set_xlabel("TC 참조 이탈 점수 > 4인 평가 창 비율 (%)")
    ax.legend(frameon=False, loc="upper right", fontsize=9)
    style(ax)
    fig.text(.025, .11, "대조 LOT도 정상 라벨이 없으며 강종 외 운전조건은 일치가 보장되지 않는다. 이 비율은 오탐률·고장 탐지 성능이 아니다.", fontsize=10)
    fig.text(.025, .06, "대조에서 변화가 흔하면 초기 참조 또는 강종만으로는 차이를 설명하지 못한다. 감지 선행 징후로 채택하기 전에 추가 검증이 필요하다.", fontsize=10)
    save(fig, output, "08_대조검토/01_과거동일강종_변화비율_대조.png", paths)


#### 그림 생성 함수

이미지는 figures/06_maintenance_timeline_jinny 아래에 저장한다. 수치표는 아래 셀 출력과 메모리에만 남긴다.


In [ ]:
def render_review(review, root):
    """Create only figures; numerical review tables remain in notebook memory."""
    validate_review(review)
    output = Path(root).resolve() / "figures" / "06_maintenance_timeline_jinny"
    paths = []
    configure_fonts()
    with plt.rc_context({"font.size": 10, "axes.unicode_minus": False}):
        coverage_figures(review, output, paths)
        boundary_figure(review, output, paths)
        observed_orders = review.coverage.groupby("WO_NO").ROWS.sum()
        for row in review.maintenance.itertuples():
            if observed_orders.loc[row.WO_NO] == 0:
                continue
            event_figure(review, row, output, paths)
            for field in POINTS:
                zoom_figure(review, row, field, output, paths)
        controls_figure(review, output, paths)
    assert all(p.is_file() and p.stat().st_size > 0 for p in paths)
    return paths


In [ ]:
# 모두 탐색 설정이다. 공학적 정상/고장 한계나 검증된 경보 문턱이 아니다.
temperature_settings = Settings(
    pre_hours=24, post_hours=24, max_gap_seconds=5,
    min_samples_per_minute=30, reference_minutes=30,
    min_reference_minutes=20, grade_settle_minutes=10,
    threshold=4.0, persistence_minutes=3, recovery_minutes=10,
    boundary_minutes=15, min_boundary_minutes=3,
)
temperature_review = load_review(ROOT / "data", temperature_settings)
validate_review(temperature_review)
temperature_figures = render_review(temperature_review, ROOT)
assert set(temperature_review.maintenance.WO_NO) == set(cal.WO_NO)
print(f"정비 {len(cal)}건 / 온도 {len(temperature_review.raw):,}행 / 추가 그림 {len(temperature_figures)}개")
display(temperature_review.coverage.pivot(index="WO_NO", columns="STAGE", values="OBSERVED_MIN")
        .reindex(columns=["감지 전", "착수 대기", "작업 중", "완료 후"])
        .rename_axis(columns="실제 인접 관측시간(분); 0은 정상/가동 의미가 아님"))


### 44건의 단계별 관측 범위

회색은 온도 관측이 없다. 색이 있더라도 실제 관측한 비율을 읽는다. 전체 대기시간이 길어도 그 전부를 관측한 것은 아니다.

![단계별 관측범위 1쪽](../figures/06_maintenance_timeline_jinny/05_온도관측범위/01_44건_단계별_관측범위.png)

![단계별 관측범위 2쪽](../figures/06_maintenance_timeline_jinny/05_온도관측범위/02_44건_단계별_관측범위.png)


## 감지·착수·완료 주변에서 무엇이 얼마나 달라졌는가?

각 경계 앞뒤 15분에서 **동일 LOT·강종**끼리 TC·OP의 1분 중앙값을 비교한다. 한쪽에 평가 가능한 분창이 3개 미만이면 차이를 계산하지 않는다.
IQR은 각 1분 안의 실제 측정값 Q75−Q25이며 온도의 흔들림을 보여준다. 단위 사전이 없어 TC ℃·OP %는 기존 표시 가정이다.

![감지·착수·완료 전후 온도와 OP 차이](../figures/06_maintenance_timeline_jinny/07_시점확대/00_감지_착수_완료_온도_OP_전후차이.png)

아래 표의 가장 큰 TC 차이는 사후 선택한 관찰이다. 냉각·출측 온도의 자연스러운 주기 변화나 강종변경도 영향을 줄 수 있어 감지 원인·정비 효과로 바로 해석하지 않는다.


In [ ]:
boundary_changes = temperature_review.boundaries.copy()
comparable_tc = boundary_changes.loc[
    boundary_changes.STATUS.eq("비교 가능") & boundary_changes.CHANNEL.isin(TC)
].copy()
comparable_tc["ABS_DELTA"] = comparable_tc.DELTA.abs()
largest_tc_changes = (comparable_tc.sort_values("ABS_DELTA", ascending=False)
                      .groupby(["WO_NO", "POINT"], sort=False).head(1))
display(largest_tc_changes[["WO_NO", "POINT", "CHANNEL", "LOT_NO", "GRADE",
                           "BEFORE_MINUTES", "AFTER_MINUTES", "BEFORE_MEDIAN", "AFTER_MEDIAN",
                           "DELTA", "BEFORE_IQR", "AFTER_IQR",
                           "BEFORE_SLOPE_PER_MIN", "AFTER_SLOPE_PER_MIN"]])
display(boundary_changes.loc[boundary_changes.STATUS.eq("비교 가능") & boundary_changes.CHANNEL.isin(OP),
    ["WO_NO", "POINT", "CHANNEL", "BEFORE_MEDIAN", "AFTER_MEDIAN", "DELTA"]])


## 감지 전에 징후가 있었고 착수 대기 중에도 이어졌는가?

설정온도가 없으므로 **감지 전 동일 LOT의 관측 구간을 참조**한다. 강종변경 직후 10분을 제외하고 첫 30분(최소 20개 유효 분창)을 사용하며, 정상이라고 확정하지 않는다.
TC별 참조 중앙값과 MAD로 점수를 구해 최대 점수가 4를 넘는 상태가 3분 이어질 때 지속변화 후보를 남긴다. 후보 시작시각과 3분 지속을 확인할 수 있는 시각을 구분한다.

- 감지까지 같은 변화 구간이 이어지고 감지 직전 공백이 2분 이내일 때만 수치상 선행시간을 표시한다. 현재 규칙에서 후보가 없다는 것은 온도 변화 자체가 없다는 뜻이 아니다.
- 대기의 **전체 길이 / 실제 관측시간 / 참조로 평가할 수 있는 분창 / 변화 후보 분창**은 다른 값이다.
- 다른 LOT·강종으로 바뀐 대기 구간에는 이전 LOT의 참조를 적용하지 않는다.
- 참조가 없거나 관측되지 않은 경우는 NA로 남긴다. 후보 창 수는 가동 중 이상 지속시간이 아니다.


In [ ]:
observed_wo = temperature_review.coverage.groupby("WO_NO").ROWS.sum()
observed_wo = observed_wo[observed_wo.gt(0)].index
event_reviews = temperature_review.decisions.loc[temperature_review.decisions.WO_NO.isin(observed_wo)]
display(event_reviews[["WO_NO", "MNT_TYPE", "WAIT_MIN", "PRE_OBSERVATION_GAP_MIN",
                       "FIRST_PRE_CHANGE", "CHANGE_CONFIRMED_AT", "CHANGE_TC", "LEAD_MIN", "LEAD_STATUS",
                       "WAIT_EVALUATED_BINS", "WAIT_CHANGE_BINS", "OTHER_WORK_ORDERS"]])
waiting_metrics = temperature_review.metrics
display(waiting_metrics.loc[
    waiting_metrics.STAGE.eq("착수 대기")
    & waiting_metrics.CHANNEL.isin(["TC-Z3", "TC-Z4", "TC-Z9", "TC-OUT1", "TC-OUT2", "OP-Z3", "OP-Z4"]),
    ["WO_NO", "LOT_NO", "GRADE", "CHANNEL", "MINUTES", "MEDIAN", "MINUTE_IQR_MEDIAN",
     "FIRST_10_MEDIAN", "LAST_10_MEDIAN", "WITHIN_SEGMENT_SHIFT"]])


### 12월 28일 감지와 대기 중 실제 변화

**WO-2024-02105:** 감지 15:54 → 착수 16:54 → 완료 23:33.
온도는 14:57~16:44:24에만 관측되어 감지 전후와 착수 대기 중 일부를 확인할 수 있다. 착수 후·완료 후 상태는 관측되지 않았다.
확대 그림은 전후 온도 중앙값 차이가 가장 큰 TC를 선택하고, 그 TC에 대응하는 OP가 없으면 보조 OP임을 명시한다.

![12월 감지 확대](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-02105_감지_확대.png)

![12월 전체 대기·작업·관측범위](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-02105_전체.png)


### 11월 13일 감지와 실제 작업이 겹치는 경우

**01859:** 온도 관측·강종변경 07:08 → 감지 07:13 → 착수 07:40 → 완료 09:35.
감지 전 관측이 5분뿐이므로 충분한 참조로 감지 선행 징후를 평가할 수 없다. 강종변경 직후의 변화도 함께 검토한다.

**01854:** 감지는 전날 15:00이지만 실제 온도는 다음 날 07:08부터 있다. 긴 대기 중 일부만 관측했다.
착수 08:54~완료 12:44로 01859와 작업이 겹친다. 두 WO의 작업·완료 시각과 온도는 함께 볼 수 있지만 개별 정비 효과는 분리할 수 없다.

![11월 감지 확대](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-01859_감지_확대.png)

![11월 작업·완료와 온도](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01854_전체.png)


### 7월 사례: 감지 전 변화가 있어도 관측이 끊긴 경우

**WO-2024-01094:** 온도는 07-08 06:29:34에 끝나고 기록된 감지는 08:49이다.
이전 LOT 구간에서 변화 후보가 나타나더라도 약 139분의 공백을 건너 감지까지 이어졌다고 말할 수 없다. 수치상 선행시간은 NA로 남긴다.

![7월 관측 공백과 감지·착수·완료](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01094_전체.png)


## 완료 후 실제 회복을 확인할 수 있는가?

감지 전 참조가 있고, 대기·작업 중 지속변화가 있었으며, 완료 후 같은 LOT·강종의 TC가 참조범위에 10분 연속 들어오는지를 확인한다.
이는 **참조범위 재진입 후보**이며 공학적 정상 회복·정비 효과의 확정이 아니다. 완료 후 공백이 있으면 정확한 회복시간을 계산하지 않는다.
감지 전 참조가 없으면 완료 전후 온도차를 관찰할 수 있어도 회복시간을 만들지 않는다.


In [ ]:
display(event_reviews[["WO_NO", "REFERENCE_STATUS", "RECOVERY_AT", "RECOVERY_MIN", "RECOVERY_STATUS",
                       "OTHER_WORK_ORDERS"]])
print("수치상 선행시간 확인 WO:", int(temperature_review.decisions.LEAD_MIN.notna().sum()))
print("완료 후 참조 재진입 시간 확인 WO:", int(temperature_review.decisions.RECOVERY_MIN.notna().sum()))


## 이 변화는 정비 주변에서만 나타나는가?

같은 참조·점수 조건을 **참조보다 과거의 동일 강종 다른 LOT**에도 적용한다. 모든 CAL01 정비의 감지 전 24시간~완료 후 24시간을 대조에서 제외한다.
동일 강종이라도 목표온도·라인속도·운전조건은 같다고 보장할 수 없으며 대조 LOT도 정상 라벨은 없다. 비율을 오탐률로 해석하지 않는다.

![과거 동일 강종 대조](../figures/06_maintenance_timeline_jinny/08_대조검토/01_과거동일강종_변화비율_대조.png)

대조에서도 변화 후보가 흔하면 초기 참조나 강종만으로 차이를 설명하기 어렵다. 이를 고장 조기 감지 규칙으로 사용하려면 목표값·운전상태·경보 원문·더 긴 연속 관측이 필요하다.


In [ ]:
display(temperature_review.controls.loc[temperature_review.controls.EVENT_MINUTES.gt(0)])
validate_review(temperature_review)
assert len(temperature_review.coverage) == 44 * 4
assert all(p.exists() and p.stat().st_size > 0 for p in temperature_figures)
print("검증: 44건·176개 단계 유지, 감지 전 참조만 사용, 공백/LOT/강종 경계 분리, 미관측 NA 유지")
print("공식 설정값·정상 라벨이 없어 분석 결과는 온도 변화와 정비 기록의 대조로 해석한다.")


## 이번 분석에서 실제 확인한 점 · 2026-10-15

- 44건 중 감지 전 24시간~완료 후 24시간에 온도 관측이 있는 작업은 10건이다. 단계별로 감지 전 8건·착수 대기 4건·작업 중 2건·완료 후 3건이다.
- 12월 28일 02105의 TC-Z9는 감지 앞뒤 15분 중앙값이 319.870 → 325.754로 +5.884 차이가 난다. 분당 추세는 +0.198 → +0.369로 달라졌다. 감지 전부터 상승하는 모습이므로 감지 시각에 갑자기 생긴 변화로 해석하지 않는다.
- 이 작업의 기록 대기는 60분이고 실제 인접 온도 관측은 약 50.4분이다. 감지 전 참조로 평가한 50개 분창 중 32개가 지속변화 후보 구간에 속한다. 전체 대기시간에 이상이 지속됐다는 뜻은 아니다.
- 같은 참조를 과거 동일 강종 대조에 적용하면 408개 분창 모두가 점수 4를 넘는다. 따라서 이 초기 참조와 문턱만으로 정상·고장을 구별할 수 없다. 강종 외 조건 차이와 참조 구간 선택을 추가 검토해야 한다.
- 11월 13일 01859는 감지 전 5분만 관측했고 강종변경 직후다. 01854와 작업도 겹쳐 온도 차이를 개별 정비 효과로 분리하기 어렵다.
- 충분한 참조와 관측 연속성을 함께 갖춘 감지 선행시간·완료 후 회복시간은 현재 자료에서 확정하지 못했다. 관측이 없거나 기준을 충족하지 못한 결과는 NA로 유지한다.

**실행 확인:** 노트북 코드 35셀을 같은 Python 프로세스에서 순서대로 실행했다. 기존 계산·원문·그림 검사 18개와 별도의 임시 수치 검증 10개가 통과했다. 새 그림 21개, 기존 원문·시간 그림 18개를 생성했다. Jupyter 커널 자동 실행 검증은 하지 않았다.

### 정비별 전체·확대 그림 찾아보기

| WO | 전체 시간축 | 감지·착수·완료 확대 |
|---|---|---|
| WO-2024-00627 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-00627_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-00717 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-00717_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-00916 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-00916_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-00972 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-00972_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-01094 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01094_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-01678 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01678_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-01854 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01854_전체.png) | [완료](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-01854_완료_확대.png) · [착수](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-01854_착수_확대.png) |
| WO-2024-01859 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01859_전체.png) | [감지](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-01859_감지_확대.png) · [완료](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-01859_완료_확대.png) · [착수](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-01859_착수_확대.png) |
| WO-2024-01922 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-01922_전체.png) | 해당 시점 주변 관측 없음 |
| WO-2024-02105 | [전체](../figures/06_maintenance_timeline_jinny/06_정비별온도대조/WO-2024-02105_전체.png) | [감지](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-02105_감지_확대.png) · [착수](../figures/06_maintenance_timeline_jinny/07_시점확대/WO-2024-02105_착수_확대.png) |


## 기존 정비 원문·작업표·타임라인 근거

아래는 44건 전체의 감지·착수·완료·REMARK·부품교체와 시간 길이를 확인하는 기존 근거다. 위 온도 대조에서 선택한 사례도 전체 44건 중 어떤 작업인지 이 표에서 확인할 수 있다.


## 3. 전체 작업표 — 원시각과 원문을 확인하는 표

**목적:** 통합 그림의 시간과 내용을 원본 필드로 다시 확인한다.
**볼 부분:** WO_NO, 정비유형, FAIL_MODE, DETC_DT·STRT_DT·END_DT, 대기·작업·정지 분,
REMARK 원문, PART_CHG와 실제 교체 부품·수량·사유.
**확인 가능한 결과:** 44건이 한 번씩 포함되는지, 감지~착수와 착수~완료가 어떻게 다른지,
REMARK가 비어 있는 작업이나 한 WO에 여러 부품이 연결된 작업을 확인한다.

11건씩 4쪽으로 나눠 글자를 키운다. REMARK 원문은 따옴표로 감싸 ""와 앞뒤 공백도 보이게 한다.
표시를 위한 줄바꿈만 추가하며 원문을 정제하거나 빈 고장모드를 채우지 않는다.


In [ ]:
work_table = cal[[
    "ORDER", "WO_NO", "MNT_TYPE", "FAIL_MODE",
    "DETC_DT", "STRT_DT", "END_DT", "WAIT_MIN", "WORK_MIN", "DOWN_MIN",
    "PART_CHG", "REMARK", "PARTS_DETAIL",
]].copy()
display(work_table)

def table_page(page, page_no, page_count):
    labels = [
        "순번 / WO_NO\n정비 / FAIL_MODE", "감지 → 착수 → 완료\n실제 날짜·시각",
        "대기 / 작업 / 정지\n각각 분", "REMARK 원문\n따옴표 안 공백도 원문",
        "교체 표기 / 부품·수량·사유\n실제 교체 시각은 제공되지 않음",
    ]
    values = []
    for row in page.itertuples():
        mode = row.FAIL_MODE if row.FAIL_MODE else "(빈 값)"
        remark = "\n".join(textwrap.wrap(literal(row.REMARK), width=23,
                    replace_whitespace=False, drop_whitespace=False)) or '""'
        parts = "\n".join(
            "\n".join(textwrap.wrap(line, width=26, replace_whitespace=False,
                                    drop_whitespace=False))
            for line in row.PARTS_DETAIL.splitlines()
        )
        values.append([
            f"{row.ORDER:02d} · {row.WO_NO}\n{row.MNT_TYPE} / {mode}",
            f"감지 {timestamp(row.DETC_DT)}\n착수 {timestamp(row.STRT_DT)}\n완료 {timestamp(row.END_DT)}",
            f"대기 {number(row.WAIT_MIN)}\n작업 {number(row.WORK_MIN)}\n정지 {number(row.DOWN_MIN)}",
            remark, f"PART_CHG={row.PART_CHG}\n{parts}",
        ])
    heights = [max(4, max(value.count("\n") + 1 for value in row)) for row in values]
    fig, ax = plt.subplots(figsize=(18, 1.6 + .30 * sum(heights)))
    ax.axis("off")
    fig.suptitle(f"CAL01 전체 작업표 · {page_no}/{page_count} · 착수 순서", x=.02, ha="left",
                 fontsize=19, fontweight="bold", y=.985)
    fig.text(.02, .947, "감지·착수·완료는 원시각 | 정지는 DOWN_MIN 길이만 확인 | REMARK·교체 원행 보존",
             fontsize=11, color="#485563")
    table = ax.table(cellText=values, colLabels=labels, cellLoc="left", colLoc="left",
                     colWidths=[.17, .24, .12, .20, .27], bbox=[0, 0, 1, .96])
    table.auto_set_font_size(False)
    table.set_fontsize(11.2)
    total_height = sum(heights) + 4
    for (ri, ci), cell in table.get_celld().items():
        cell.set_edgecolor("#d6dde3")
        cell.PAD = .045
        if ri == 0:
            cell.set_facecolor("#e6edf3")
            cell.set_text_props(weight="bold")
            cell.set_height(4 / total_height)
        else:
            cell.set_height(heights[ri - 1] / total_height)
            cell.set_facecolor("#f5f7fa" if ri % 2 == 0 else "white")
            if ci == 0:
                cell.get_text().set_color(COLORS.get(page.iloc[ri - 1].MNT_TYPE, "#333333"))
    fig.legend(handles=[Line2D([], [], color=COLORS[k], lw=5, label=f"{k} {TYPE_NAMES[k]}") for k in ["BM", "CBM", "TBM"]], loc="upper left", bbox_to_anchor=(.015, .921), ncol=3, frameon=False, fontsize=11)
    fig.text(.02, .018, "(a) 감지·착수·완료는 원본 시각이다. 따옴표 안 REMARK 공백과 빈 값도 원문이며, DOWN_MIN은 정지 길이만 뜻한다.", fontsize=10.5, color="#536779")
    fig.subplots_adjust(left=.02, right=.98, top=.875, bottom=.065)
    save_figure(fig, f"01_전체작업표/{page_no:02d}_전체작업표_순번{int(page.ORDER.min()):02d}-{int(page.ORDER.max()):02d}.png")

page_size = 11
page_count = math.ceil(len(cal) / page_size)
table_covered = []
for page_no, start in enumerate(range(0, len(cal), page_size), 1):
    page = cal.iloc[start:start + page_size]
    table_covered.extend(page.WO_NO.tolist())
    table_page(page, page_no, page_count)
assert table_covered == cal.WO_NO.tolist(), "44건 작업표의 중복·누락 확인"
display(cal_parts[["SOURCE_CSV_ROW", "WO_NO", "PART_CD", "PART_NM",
                   "PART_TYPE", "QTY", "CHG_RSN", "EXACT_DUPLICATE", "MASTER_LINK"]])


## 4. 월별 통합 타임라인 — 정비 시각과 작업 내용을 한 줄에서 확인

**목적:** 제공된 예시처럼 작업 1건을 한 줄로 놓고 정비 시각·내용·교체·온도 자료의 시간 관계를 함께 확인한다.
**볼 부분:** 월별 정비 건수, 정비유형 색, 감지→착수→완료, 실제 대기/작업/정지 분,
REMARK 원문과 FAIL_MODE, 교체 부품, 작업 중 온도 측정 행, 감지 시각에서 ±72시간 이내인 LOT.
**확인 가능한 결과:** 가장 오래 대기한 WO, 정지 0인 작업, 부품이 여러 개 연결된 작업,
정기수리 캘린더와 겹친 작업 및 정비 중 센서 관측 가능 여부를 직접 대조한다.

전체 44건 그림과 같은 내용을 확대한 분기별 4개 그림을 저장한다. **감지 시각 순서**이며 WO 번호순이 아니다.
작은 타임라인은 감지를 기준으로 한 작업별 순서를 나타낸다.
대기가 6시간을 넘으면 점선 가운데 //로 축 생략을 표시한다. **생략 전 실제 대기 분은 별도 열에 그대로 표기**한다.
작업 막대 길이는 모든 행에 동일한 시간 배율을 쓴다. 실제 감지·착수·완료 날짜와 시각은 전체 작업표와 작업 상세에 남긴다.

온도 기록은 실제 MEAS_DT 행을 검사한다. 작업 중은 [STRT_DT, END_DT), 감지 시각은 DETC_DT와 정확히 일치한 행이다.
LOT의 최소·최대 시각 사이를 보간하지 않는다. ±72시간은 감지 시각과 LOT 시작·종료의 실제 거리이며,
시간이 가깝다는 사실을 소재 연결이나 정비 효과로 해석하지 않는다.

### 원본 대조 결과

- CAL01 44건: BM 12건, CBM 10건, TBM 22건. 부품교체는 8 WO에 9행이 연결된다.
- 작업 중 온도 측정 행이 있는 WO: WO-2024-01859, WO-2024-01854. 감지 시각에는 있지만 작업 중에는 없는 WO: WO-2024-02105.
- 최장 대기 WO-2024-00627는 12,983분이다. //는 그림의 축 생략 표시이며 실제 대기값은 이 숫자로 남는다.
- CR1 캘린더와 작업 구간이 겹치는 WO는 11건이다. 실제 정지 구간으로 해석하지 않는다.

In [ ]:
# 예시와 같은 한 행 한 작업 통합 그림. 계산값은 그대로 두고 긴 대기 표시에서만 축을 생략한다.
WAIT_DISPLAY_LIMIT_H = 6
timeline_files = []
overview_layout = []
def ledger_fields(row, fig_width, fontsize):
    values = [
        row.DETC_DT.strftime("%m/%d %H:%M"), row.MNT_TYPE, row.WO_NO,
        "", number(row.WAIT_MIN), number(row.WORK_MIN), number(row.DOWN_MIN),
        literal(row.REMARK) + "\nFAIL_MODE=" + literal(row.FAIL_MODE),
        row.PARTS_DETAIL, row.TEMP_CONTEXT, row.NEARBY_LOTS,
    ]
    widths = np.diff(LEDGER_EDGES)
    return [wrap_units(value, max(8, int(width * fig_width * .95 * 72 / (fontsize * .55)) - 2))
            if i not in [1, 3] else value for i, (value, width) in enumerate(zip(values, widths))]

LEDGER_EDGES = np.array([0, .105, .15, .235, .385, .438, .493, .546, .704, .835, .895, 1])
LEDGER_HEADERS = ["감지 시각\n2024년", "정비\n유형", "WO_NO", "◆ 감지 · 점선 대기 · 막대 작업",
                  "대기시간\n분", "작업시간\n분", "정지시간\nDOWN_MIN·분",
                  "REMARK 원문\nFAIL_MODE", "교체 부품·수량·사유", "온도 기록", "가까운 LOT\n감지 기준 ±72시간"]
chronology = cal.sort_values(["DETC_DT", "STRT_DT", "WO_NO"]).copy()

def integrated_timeline(data, filename, suffix, count_coverage=False):
    fw, fontsize, unit = 22, 10.8, .205
    entries = []
    for month, group in data.groupby(data.DETC_DT.dt.month, sort=True):
        entries.append(("month", month, 1.3, None))
        for row in group.itertuples():
            fields = ledger_fields(row, fw, fontsize)
            height = max(2.1, max(text.count("\n") + 1 for text in fields) * 1.05 + .5)
            entries.append(("task", row, height, fields))
            if count_coverage:
                timeline_covered.append(row.WO_NO)
    total = sum(item[2] for item in entries)
    top, bottom = 1.90, 1.12
    height = top + bottom + total * unit
    fig = plt.figure(figsize=(fw, height))
    ax = fig.add_axes([.025, bottom / height, .95, total * unit / height])
    ax.set(xlim=(0, 1), ylim=(total, 0))
    ax.axis("off")
    fig.text(.025, 1 - .24 / height, f"CAL01 정비 이력 {len(data)}건 · 감지 → 착수 → 완료와 작업 내용 · {suffix}",
             fontsize=18, weight="bold", va="top", color="#233F53")
    fig.text(.025, 1 - .65 / height,
             f"작업 1건 = 1행 · 감지 시각순 · 실제 시간 길이는 분 단위 · 점선 대기 6시간 초과는 // 축 생략",
             fontsize=11, va="top", color="#536779")
    handles = [Line2D([], [], lw=5, color=COLORS[k], label=f"{k} {TYPE_NAMES[k]} ({int(data.MNT_TYPE.eq(k).sum())}건)")
               for k in ["BM", "CBM", "TBM"]]
    handles += [Line2D([], [], color=PART_COLOR, marker="s", linestyle="", label="부품교체 기록"),
                Line2D([], [], color=TEMP_COLOR, marker="s", linestyle="", label="작업 중 온도 기록")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.022, 1 - .91 / height),
               ncol=5, fontsize=10.5, frameon=False)
    for i, header in enumerate(LEDGER_HEADERS):
        fig.text(.025 + .95 * (LEDGER_EDGES[i] + .004), 1 - 1.36 / height, header,
                 fontsize=10.3, va="top", color="#456276", weight="bold", linespacing=1.25)
    y = 0
    max_x_hours = WAIT_DISPLAY_LIMIT_H + cal.WORK_MIN.max() / 60
    mini_left, mini_right = LEDGER_EDGES[3] + .007, LEDGER_EDGES[4] - .012
    scale = (mini_right - mini_left) / max_x_hours
    for kind, value, row_height, fields in entries:
        center = y + row_height / 2
        if kind == "month":
            group = data.loc[data.DETC_DT.dt.month.eq(value)]
            plans = cr1_plans.loc[cr1_plans.START.dt.month.eq(value)]
            plan_label = " · ".join(
                f"CR1 정기수리 {p.STRT_DT[5:]}~{p.END_DT[5:]} ({p.SCOPE})" for p in plans.itertuples())
            ax.add_patch(Rectangle((0, y + .08), 1, row_height - .16, facecolor="#E7EEF4", edgecolor="none"))
            ax.text(.004, center, f"{value}월 · {len(group)}건" + (" · " + plan_label if plan_label else ""),
                    va="center", fontsize=11, weight="bold", color="#29495E")
        else:
            row = value
            color = COLORS[row.MNT_TYPE]
            if row.PLAN_IDS:
                ax.add_patch(Rectangle((0, y + .035), 1, row_height - .07,
                                       facecolor=PLAN_BG, edgecolor="none", zorder=0))
            ax.add_patch(Rectangle((0, y + .18), .0025, row_height - .36,
                                   facecolor=color, edgecolor="none"))
            for col, text in enumerate(fields):
                if col == 3:
                    continue
                text_color = (color if col == 1 else PART_COLOR if col == 8 and row.PART_ROWS > 0
                              else TEMP_COLOR if col == 9 and row.TEMP_WORK_ROWS > 0 else "#334E60")
                artist = ax.text(LEDGER_EDGES[col] + .004, center, text, va="center",
                                 fontsize=fontsize if col not in [2, 10] else fontsize - .5,
                                 color=text_color, linespacing=1.25,
                                 weight="bold" if col in [1, 9] and (col == 1 or row.TEMP_WORK_ROWS > 0) else "normal")
                overview_layout.append((filename, artist, LEDGER_EDGES[col + 1] - .003, y, y + row_height))
            displayed_wait = min(row.WAIT_MIN / 60, WAIT_DISPLAY_LIMIT_H)
            start_x = mini_left + displayed_wait * scale
            end_x = start_x + row.WORK_MIN / 60 * scale
            ax.plot([mini_left, start_x], [center, center], linestyle=(0, (1, 2)), color=color, lw=1.6)
            ax.plot([start_x, end_x], [center, center], color=color, lw=6, solid_capstyle="butt")
            ax.scatter([mini_left], [center], marker="D", color=color, s=22, zorder=4)
            if row.WAIT_MIN / 60 > WAIT_DISPLAY_LIMIT_H:
                ax.text(mini_left + displayed_wait * scale * .68, center, "//", color=color,
                        fontsize=12, ha="center", va="center",
                        bbox={"facecolor":PLAN_BG if row.PLAN_IDS else "white", "edgecolor":"none", "pad":.25})
        y += row_height
    fig.text(.025, .73 / height,
             "(a) 점선은 감지~착수 대기, 실선 막대는 착수~완료 작업. //는 대기 6시간 초과 부분의 축 생략이며 오른쪽 대기시간은 원본 시각 차이 전체다.",
             fontsize=10, color="#536779", va="top")
    fig.text(.025, .49 / height,
             "(b) 온도 기록은 실제 측정 행의 존재만 뜻한다. 작업 중=[착수, 완료), 감지 시각=원본 DETC_DT 정확 일치. 온도값의 정상·이상 판정은 하지 않는다.",
             fontsize=10, color="#536779", va="top")
    fig.text(.025, .25 / height,
             "(c) 옅은 행 배경은 CR1 정기수리 캘린더와 작업 구간의 겹침. DOWN_MIN은 길이만 표시하며 정지 구간을 만들어 표시하지 않는다.",
             fontsize=10, color="#536779", va="top")
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for file, artist, right, low, high in overview_layout:
        if file != filename:
            continue
        points = ax.transData.inverted().transform(artist.get_window_extent(renderer).get_points())
        x0, y0 = points.min(axis=0)
        x1, y1 = points.max(axis=0)
        if x1 > right + .002 or y0 < low - .04 or y1 > high + .04:
            layout_issues.append((filename, "열/행 경계: " + artist.get_text()[:70]))
    save_figure(fig, filename)
    timeline_files.append(filename)

integrated_timeline(chronology, "02_통합타임라인/01_CAL01_정비44건_전체.png", "전체")
for quarter in range(1, 5):
    lo, hi = (quarter - 1) * 3 + 1, quarter * 3
    data = chronology.loc[chronology.DETC_DT.dt.month.between(lo, hi)]
    integrated_timeline(data, f"02_통합타임라인/{quarter+1:02d}_{lo:02d}-{hi:02d}월_정비기록.png",
                        f"{lo}~{hi}월 확대", count_coverage=True)
assert timeline_covered == chronology.WO_NO.tolist()


## 5. 대기시간·작업시간·정지시간 비교

**목적:** 작업별 감지~착수 대기시간, 착수~완료 작업시간, 원본 DOWN_MIN의 차이를 비교한다.
**볼 부분:** 왼쪽 최장 대기 WO와 오른쪽 최대 DOWN_MIN WO, 작업시간=DOWN_MIN 점선에서 벗어난 점과 정지 0인 점.
**확인 가능한 결과:** 긴 대기가 긴 작업·정지와 항상 같은지를 확인하고, 작업시간과 정지 길이가 다른 작업을 찾는다.

각 점은 WO 1건이며 BM·CBM·TBM 색은 통합 그림과 같다.
왼쪽 대기시간 축은 60분까지 선형, 이후 로그 축임을 축과 왼쪽 아래 (a)에 적는다.
DOWN_MIN을 달력상의 정지 구간이나 연간 정지 총량으로 바꾸지 않는다.

### 원본 대조 결과

- WO-2024-01346: DOWN_MIN 610분, 작업시간 1,044분으로 다르다.
- DOWN_MIN=0인 작업은 7건이다. 0은 원본값이며 무효값으로 채우거나 삭제하지 않았다.

In [ ]:
duration_summary = cal.groupby("MNT_TYPE").agg(
    건수=("WO_NO", "size"), 대기중앙값분=("WAIT_MIN", "median"),
    대기최대분=("WAIT_MIN", "max"), 작업중앙값분=("WORK_MIN", "median"),
    작업최대분=("WORK_MIN", "max"), 정지중앙값분=("DOWN_MIN", "median"),
    정지최대분=("DOWN_MIN", "max"), 정지0건수=("DOWN_MIN", lambda s: s.eq(0).sum()),
)
display(duration_summary)
fig, axs = plt.subplots(1, 2, figsize=(16, 7.5))
fig.subplots_adjust(left=.07, right=.97, top=.80, bottom=.19, wspace=.26)
fig.suptitle("CAL01 · 대기시간, 작업시간, 정지시간은 서로 다른 값", x=.03, ha="left",
             fontsize=19, fontweight="bold", y=.97)
fig.text(.03, .90, "각 점은 WO 1건 | 유형 색은 동일 | 정지 0도 원본값으로 유지", fontsize=11)
for kind, group in cal.groupby("MNT_TYPE"):
    axs[0].scatter(group.WAIT_MIN, group.WORK_MIN, label=f"{kind} ({len(group)}건)",
                   s=45, color=COLORS.get(kind, "#555555"), alpha=.8, edgecolor="white", linewidth=.4)
    axs[1].scatter(group.WORK_MIN, group.DOWN_MIN, s=45,
                   color=COLORS.get(kind, "#555555"), alpha=.8, edgecolor="white", linewidth=.4)
axs[0].set_xscale("symlog", linthresh=60)
axs[0].set_xlabel("감지~착수 대기시간 (분; 60분 이후 로그 축)")
axs[0].set_ylabel("착수~완료 작업시간 (분)")
axs[0].set_title("대기가 길어도 정지시간의 위치·길이는 별개", loc="left")
limit = max(cal.WORK_MIN.max(), cal.DOWN_MIN.max()) * 1.08
axs[1].plot([0, limit], [0, limit], "--", color=WAIT_COLOR, lw=1, label="작업시간 = DOWN_MIN")
axs[1].set(xlim=(-25, limit), ylim=(-25, limit),
           xlabel="착수~완료 작업시간 (분)", ylabel="정지시간 DOWN_MIN (분)")
axs[1].set_title("작업시간과 정지 길이가 같은지 대조", loc="left")
for ax in axs:
    ax.grid(alpha=.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="upper left")
for wo, reason in [(cal.loc[cal.WAIT_MIN.idxmax(), "WO_NO"], "최장 대기"),
                   (cal.loc[cal.DOWN_MIN.idxmax(), "WO_NO"], "최대 정지")]:
    row = cal.loc[cal.WO_NO.eq(wo)].iloc[0]
    target, x, yval = (axs[0], row.WAIT_MIN, row.WORK_MIN) if reason == "최장 대기" else (axs[1], row.WORK_MIN, row.DOWN_MIN)
    target.annotate(f"{reason}\n{wo}", (x, yval), xytext=(-8, 18), textcoords="offset points",
                    ha="right", fontsize=10.5, arrowprops={"arrowstyle":"-", "color":"#465563"})
fig.text(.03, .035, "(a) 같은 색은 같은 정비유형. 왼쪽 대기시간 축은 60분까지 선형, 이후 로그 축이다. 오른쪽 점선은 작업시간=DOWN_MIN이며 정지 시작·종료 시각을 뜻하지 않는다.", fontsize=10, color="#536779")
axs[1].set_xticks(np.arange(0, limit + .01, 200))
axs[1].set_yticks(np.arange(0, limit + .01, 200))
save_figure(fig, "03_시간비교/01_대기시간_작업시간_정지시간_비교.png")


## 6. 동시 작업과 관심 사례 선정

같은 설비의 작업 구간을 **[착수, 완료)**로 비교한다. 한 작업 완료와 다음 착수가 같은 경우는 겹침으로 세지 않는다.
감지 시각이 작업 밖에 있다는 이유로 다른 WO와 합치지 않는다.
확대 사례는 최대 대기·최대 정지·정지 0 중 최장 작업·복수 부품교체·모든 동시 작업 쌍·
7월 절연 저하 기록·12월 교체 기록을 명시적인 규칙으로 선택한다.


In [ ]:
overlap_rows = []
for i, left in cal.iterrows():
    for _, right in cal.iloc[i + 1:].iterrows():
        if pd.isna(left.STRT_DT) or pd.isna(left.END_DT) or pd.isna(right.STRT_DT) or pd.isna(right.END_DT):
            continue
        start, end = max(left.STRT_DT, right.STRT_DT), min(left.END_DT, right.END_DT)
        if end > start:
            overlap_rows.append({
                "WO_A": left.WO_NO, "WO_B": right.WO_NO, "OVERLAP_START": start,
                "OVERLAP_END": end, "OVERLAP_MIN": (end - start).total_seconds() / 60,
            })
overlaps = pd.DataFrame(overlap_rows, columns=[
    "WO_A", "WO_B", "OVERLAP_START", "OVERLAP_END", "OVERLAP_MIN"])
display(overlaps)

cases = []
def add_case(key, title, wo_numbers, reason):
    selected = cal.loc[cal.WO_NO.isin(wo_numbers)]
    assert len(selected) == len(set(wo_numbers)), f"사례 WO 확인: {wo_numbers}"
    cases.append({"CASE": key, "TITLE": title, "WO_LIST": list(selected.WO_NO), "REASON": reason})

add_case("A", "최장 대기", [cal.loc[cal.WAIT_MIN.idxmax(), "WO_NO"]],
         "감지~착수 대기시간 최대; 대기와 정지 길이를 분리")
add_case("B", "최대 정지 길이", [cal.loc[cal.DOWN_MIN.idxmax(), "WO_NO"]],
         "DOWN_MIN 최대; 작업시간과 정지 길이의 차이")
zero = cal.loc[cal.DOWN_MIN.eq(0)].sort_values(["WORK_MIN", "WO_NO"], ascending=[False, True])
if not zero.empty:
    add_case("C", "정지 0 중 최장 작업", [zero.iloc[0].WO_NO],
             "DOWN_MIN=0이고 작업시간이 가장 긴 WO; 운전상태 추정 금지")
multi = cal.loc[cal.PART_ROWS.gt(1)]
if not multi.empty:
    add_case("D", "한 WO의 여러 교체 부품", [multi.iloc[0].WO_NO],
             "일대다 교체 연결을 실제 부품·수량·사유로 확인")
for i, pair in enumerate(overlaps.itertuples(), 1):
    add_case(f"E{i}", f"{pair.OVERLAP_START:%m월 %d일} 작업 구간 겹침", [pair.WO_A, pair.WO_B],
             f"작업 교집합 {number(pair.OVERLAP_MIN)}분; WO별 REMARK·정비유형 분리")
for key, wo, title in [
    ("F", "WO-2024-01094", "7월 절연 저하·교체 기록"),
    ("G", "WO-2024-02105", "12월 감지·착수·교체 기록"),
]:
    if cal.WO_NO.eq(wo).any():
        add_case(key, title, [wo], "05번에서 관심을 둔 WO의 세 시각·원문·부품을 독립 대조")
case_table = pd.DataFrame([
    {"사례": c["CASE"], "주제": c["TITLE"], "대상 WO": ", ".join(c["WO_LIST"]), "선정 근거": c["REASON"]}
    for c in cases
])
display(case_table)

CASE_NAMES = {"A": "최장대기", "B": "최대정지시간", "C": "정지0_최장작업", "D": "복수부품교체",
              "E1": "11월09일_작업겹침", "E2": "11월13일_작업겹침",
              "F": "07월08일_절연저하_교체", "G": "12월28일_마모량초과_교체"}
for i, case in enumerate(cases, 1):
    case["FILENAME"] = f"04_관심작업상세/{i:02d}_{CASE_NAMES[case['CASE']]}_{'_'.join(case['WO_LIST'])}.png"


## 7. 관심 작업 상세

**목적:** 통합 그림에서 눈에 띄는 작업을 실제 감지·착수·완료 시각과 원문으로 확인한다.
각 그림은 (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN 비교로 구성한다.
오른쪽에는 해당 WO의 세 원시각·대기시간·작업시간·DOWN_MIN·REMARK 원문·교체 부품을 적는다.
긴 대기시간은 오른쪽에 실제 길이로 표시하고, 작업시간과 DOWN_MIN 비교 축은 두 값만 사용한다.
설명이 필요한 항목은 그림 왼쪽 아래 (a)~(c)에 적는다.
두 작업이 겹치면 실제 작업 교집합만 옅은 노랑으로 표시한다. 부품교체 시각은 만들지 않는다.


In [ ]:
def draw_task(ax, row, y):
    color = COLORS[row.MNT_TYPE]
    ax.plot([row.DETC_DT, row.STRT_DT], [y, y], linestyle=(0, (2, 2)), color=color, lw=1.8)
    ax.plot([row.STRT_DT, row.END_DT], [y, y], color=color, lw=7, solid_capstyle="butt")
    for field, marker in [("DETC_DT", "D"), ("STRT_DT", "o"), ("END_DT", "s")]:
        ax.scatter([getattr(row, field)], [y], color=color, marker=marker,
                   s=55, edgecolor="white", linewidth=.7, zorder=4)

def case_figure(case):
    selected = cal.loc[cal.WO_NO.isin(case["WO_LIST"])].copy()
    rows = list(selected.itertuples())
    count = len(rows)
    blocks = []
    for row in rows:
        lines = [
            f"{row.WO_NO} · {row.MNT_TYPE} {TYPE_NAMES[row.MNT_TYPE]}",
            "FAIL_MODE = " + literal(row.FAIL_MODE),
            f"감지 {timestamp(row.DETC_DT)}",
            f"착수 {timestamp(row.STRT_DT)}",
            f"완료 {timestamp(row.END_DT)}",
            f"대기 {number(row.WAIT_MIN)}분 ({duration_text(row.WAIT_MIN)})",
            f"작업 {number(row.WORK_MIN)}분 · 정지 {number(row.DOWN_MIN)}분",
            "REMARK 원문 = " + literal(row.REMARK),
            "부품교체 = " + row.PART_CHG,
            row.PARTS_DETAIL,
            f"작업 중 온도 측정 행 = {number(row.TEMP_WORK_ROWS)}행",
        ]
        blocks.append(wrap_units("\n".join(lines), 55))
    block_heights = [(text.count("\n") + 1) * .225 + .35 for text in blocks]
    body_height = max(5.0, sum(block_heights) + .25 * (count - 1))
    height = body_height + 3.25
    fig = plt.figure(figsize=(16, height))
    grid = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], width_ratios=[1.8, 1.05],
                           left=.13, right=.97, top=1 - 1.85 / height, bottom=1.35 / height,
                           hspace=.95, wspace=.32)
    whole = fig.add_subplot(grid[0, 0])
    work = fig.add_subplot(grid[1, 0])
    lengths = fig.add_subplot(grid[2, 0])
    card = fig.add_subplot(grid[:, 1])
    card.axis("off")
    fig.text(.025, 1 - .22 / height, f"CAL01 · {case['TITLE']}", fontsize=20,
             weight="bold", color="#233F53", va="top")
    fig.text(.025, 1 - .65 / height, case["REASON"], fontsize=11.5, color="#536779", va="top")
    handles = [Line2D([], [], color=COLORS[k], lw=5, label=f"{k} {TYPE_NAMES[k]}")
               for k in selected.MNT_TYPE.unique()]
    handles += [Line2D([], [], color=WAIT_COLOR, linestyle="--", label="감지~착수 대기"),
                Line2D([], [], color="#405565", marker="D", linestyle="", label="감지"),
                Line2D([], [], color="#405565", marker="o", linestyle="", label="착수"),
                Line2D([], [], color="#405565", marker="s", linestyle="", label="완료"),
                Line2D([], [], color=DOWN_COLOR, lw=5, label="정지시간 DOWN_MIN")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.022, 1 - .92 / height),
               ncol=7, fontsize=10, frameon=False)
    lo = selected[["DETC_DT", "STRT_DT", "END_DT"]].min().min()
    hi = selected[["DETC_DT", "STRT_DT", "END_DT"]].max().max()
    work_lo, work_hi = selected.STRT_DT.min(), selected.END_DT.max()
    for i, row in enumerate(rows):
        draw_task(whole, row, i)
        draw_task(work, row, i)
        lengths.barh(i - .17, row.WORK_MIN, height=.26, color=COLORS[row.MNT_TYPE])
        lengths.barh(i + .17, row.DOWN_MIN, height=.26, color=DOWN_COLOR)
        for offset, value, label in [(-.17, row.WORK_MIN, "작업"), (.17, row.DOWN_MIN, "정지")]:
            lengths.annotate(f"{label} {number(value)}분", (value, i + offset), xytext=(6, 0),
                             textcoords="offset points", va="center", fontsize=10.5)
    for ax in (whole, work, lengths):
        ax.set_yticks(range(count), [row.WO_NO for row in rows], fontsize=10)
        ax.set_ylim(count - .55, -.55)
    whole_pad = max((hi - lo) * .06, pd.Timedelta(minutes=8))
    work_pad = max((work_hi - work_lo) * .07, pd.Timedelta(minutes=8))
    whole.set_xlim(lo - whole_pad, hi + whole_pad)
    work.set_xlim(work_lo - work_pad, work_hi + work_pad)
    whole.set_title("(a) 감지 → 착수 → 완료 · 전체", loc="left", pad=9)
    work.set_title("(b) 착수 → 완료 · 작업 구간 확대", loc="left", pad=9)
    for ax in (whole, work):
        actual_clock(ax)
    lengths.set_title("(c) 작업시간과 정지시간 비교", loc="left", pad=9)
    lengths.set_xlim(0, max(selected.WORK_MIN.max(), selected.DOWN_MIN.max(), 1) * 1.46)
    lengths.set_xlabel("시간 길이 (분)")
    lengths.grid(axis="x", alpha=.18)
    lengths.spines[["top", "right"]].set_visible(False)
    for pair in overlaps.itertuples():
        if pair.WO_A in case["WO_LIST"] and pair.WO_B in case["WO_LIST"]:
            for ax in (whole, work):
                ax.axvspan(pair.OVERLAP_START, pair.OVERLAP_END, color="#E8B74D", alpha=.18)
                ax.text(.99, .97, f"작업 겹침 {number(pair.OVERLAP_MIN)}분", transform=ax.transAxes,
                        ha="right", va="top", fontsize=10.5, color="#754F00")
    card_height = card.get_position().height * height
    cursor = 1.0
    for i, (row, text, block_height) in enumerate(zip(rows, blocks, block_heights)):
        fraction = block_height / card_height
        card.add_patch(Rectangle((0, cursor - fraction), 1, fraction, transform=card.transAxes,
                                  facecolor="#F5F7F9", edgecolor="#DFE6EC", lw=.8))
        card.add_patch(Rectangle((0, cursor - fraction), .012, fraction, transform=card.transAxes,
                                  facecolor=COLORS[row.MNT_TYPE], edgecolor="none"))
        card.text(.04, cursor - .015, text, transform=card.transAxes, va="top", fontsize=11.5,
                  color="#263F50", linespacing=1.4)
        cursor -= fraction + .25 / card_height
    has_overlap = any(p.WO_A in case["WO_LIST"] and p.WO_B in case["WO_LIST"] for p in overlaps.itertuples())
    notes = [
        "(a) 실제 기록 시각이다. 감지~착수는 점선, 착수~완료는 작업 막대다.",
        "(b) 옅은 노랑은 두 WO의 실제 작업 구간 교집합이다." if has_overlap else "(b) 작업 확대는 원본 착수~완료 구간이다. 부품교체 시각은 원본에 없다.",
        "(c) DOWN_MIN은 정지시간의 길이다. 정지 시작·종료 시각이나 정지 구간으로 바꾸지 않는다.",
    ]
    for y, note in zip([.65, .42, .19], notes):
        fig.text(.025, y / height, note, fontsize=10, color="#536779", va="top")
    save_figure(fig, case["FILENAME"])


### 최장 대기

**목적:** 감지~착수 대기시간 최대; 대기와 정지 길이를 분리.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-00627 · CBM: 대기 12,983분, 작업 182분, DOWN_MIN 105분. REMARK 원문="확인 중".
  부품교체: 교체 기록 없음. 작업 중 온도 측정 행 0행.

저장 그림: [01_최장대기_WO-2024-00627.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/01_최장대기_WO-2024-00627.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "A"))


### 최대 정지시간

**목적:** DOWN_MIN 최대; 작업시간과 정지 길이의 차이.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-01346 · BM: 대기 206분, 작업 1,044분, DOWN_MIN 610분. REMARK 원문="1호기 재발, 모터 과열".
  부품교체: SEL-006 오일씰 150x190x18 ×1 / 손상. 작업 중 온도 측정 행 0행.

저장 그림: [02_최대정지시간_WO-2024-01346.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/02_최대정지시간_WO-2024-01346.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "B"))


### 정지 0 중 최장 작업

**목적:** DOWN_MIN=0이고 작업시간이 가장 긴 WO; 운전상태 추정 금지.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-01467 · CBM: 대기 9,494분, 작업 471분, DOWN_MIN 0분. REMARK 원문="  재점검 필요 ".
  부품교체: 교체 기록 없음. 작업 중 온도 측정 행 0행.

저장 그림: [03_정지0_최장작업_WO-2024-01467.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/03_정지0_최장작업_WO-2024-01467.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "C"))


### 한 WO의 복수 부품교체

**목적:** 일대다 교체 연결을 실제 부품·수량·사유로 확인.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-00916 · TBM: 대기 142분, 작업 123분, DOWN_MIN 122분. REMARK 원문="".
  부품교체: SEL-006 오일씰 150x190x18 ×2 / 수명도래; SEL-009 O링 세트 ×1 / 손상. 작업 중 온도 측정 행 0행.

저장 그림: [04_복수부품교체_WO-2024-00916.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/04_복수부품교체_WO-2024-00916.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "D"))


### 11월 9일 실제 작업 겹침

**목적:** 작업 교집합 107분; WO별 REMARK·정비유형 분리.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-01823 · TBM: 대기 233분, 작업 184분, DOWN_MIN 70분. REMARK 원문="1호기 재발, 정기수리 실시".
  부품교체: 교체 기록 없음. 작업 중 온도 측정 행 0행.
- WO-2024-01826 · TBM: 대기 216분, 작업 425분, DOWN_MIN 425분. REMARK 원문="수리기간 중 점검".
  부품교체: 교체 기록 없음. 작업 중 온도 측정 행 0행.
- 두 작업의 실제 교집합: 2024-11-09 11:15~13:02, 107분.

저장 그림: [05_11월09일_작업겹침_WO-2024-01823_WO-2024-01826.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/05_11월09일_작업겹침_WO-2024-01823_WO-2024-01826.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "E1"))


### 11월 13일 실제 작업 겹침

**목적:** 작업 교집합 41분; WO별 REMARK·정비유형 분리.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-01859 · TBM: 대기 27분, 작업 115분, DOWN_MIN 115분. REMARK 원문="계획 점검 수행".
  부품교체: 교체 기록 없음. 작업 중 온도 측정 행 6,637행.
- WO-2024-01854 · CBM: 대기 1,074분, 작업 230분, DOWN_MIN 190분. REMARK 원문="마모량 초과".
  부품교체: 교체 기록 없음. 작업 중 온도 측정 행 13,328행.
- 두 작업의 실제 교집합: 2024-11-13 08:54~09:35, 41분.

저장 그림: [06_11월13일_작업겹침_WO-2024-01859_WO-2024-01854.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/06_11월13일_작업겹침_WO-2024-01859_WO-2024-01854.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "E2"))


### 7월 8일 절연 저하·교체 기록

**목적:** 05번에서 관심을 둔 WO의 세 시각·원문·부품을 독립 대조.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-01094 · BM: 대기 142분, 작업 473분, DOWN_MIN 405분. REMARK 원문="1호기 재발, 절연 저하".
  부품교체: SEL-009 O링 세트 ×3 / 예방. 작업 중 온도 측정 행 0행.

저장 그림: [07_07월08일_절연저하_교체_WO-2024-01094.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/07_07월08일_절연저하_교체_WO-2024-01094.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "F"))


### 12월 28일 마모량 초과·교체 기록

**목적:** 05번에서 관심을 둔 WO의 세 시각·원문·부품을 독립 대조.
**볼 부분:** (a) 감지~완료 전체, (b) 작업 구간 확대, (c) 작업시간과 DOWN_MIN, 오른쪽 원시각·REMARK·교체 부품.
**확인 결과:**

- WO-2024-02105 · CBM: 대기 60분, 작업 399분, DOWN_MIN 330분. REMARK 원문="마모량 초과".
  부품교체: INS-006 가스 분석계 셀 ×3 / 예방. 작업 중 온도 측정 행 0행.

저장 그림: [08_12월28일_마모량초과_교체_WO-2024-02105.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/08_12월28일_마모량초과_교체_WO-2024-02105.png)

In [ ]:
case_figure(next(case for case in cases if case["CASE"] == "G"))


## 8. 계산·원문·시각화 확인

44건 유지, 원문 보존, 시간 차이, 부품 연결, 작업 겹침, 그림 누락·중복을 실제 값으로 검증한다.
통합 그림은 글자가 행·열 경계를 넘는지, 모든 그림은 글자가 이미지 경계를 넘는지도 확인한다.
범례·색·작업 막대·긴 대기 축 생략·작업 겹침 음영은 생성된 그림을 열어 별도로 확인한다.
시각화는 원본 값을 바꾸거나 정지 시각·부품교체 시각·고장 원인·교체 효과를 만들어내지 않는다.


In [ ]:
checks = []
def checked(name, condition, evidence):
    assert bool(condition), name
    checks.append({"검증": name, "결과": "통과", "근거": evidence})

checked("CAL01 범위·건수", len(cal) == 44 and cal.EQP_TAG.eq(EQP_TAG).all(), f"{len(cal)}건")
checked("WO당 작업표 한 행", cal.WO_NO.is_unique and len(work_table) == len(cal), "일대다 교체로 작업 행 증가 없음")
original = cal_raw.set_index("WO_NO")
joined = cal.set_index("WO_NO").reindex(original.index)
checked("REMARK 원문 보존", joined.REMARK.equals(original.REMARK), "공백·오타·빈 문자열 그대로")
checked("FAIL_MODE 원문 보존", joined.FAIL_MODE.equals(original.FAIL_MODE), "결측을 고장모드로 채우지 않음")
for field in ["DETC_DT", "STRT_DT", "END_DT"]:
    checked(field + " 원문 보존", joined[field + "_RAW"].equals(original[field]), "분석용 datetime과 원문을 별도 보유")
checked("시각 파싱과 시간 순서", cal[["DETC_DT", "STRT_DT", "END_DT"]].notna().all().all()
        and cal.WAIT_MIN.ge(0).all() and cal.WORK_MIN.ge(0).all(), "현재 44건의 시각을 실제 확인")
checked("대기+작업=감지~완료", np.allclose(cal.WAIT_MIN + cal.WORK_MIN, cal.DETECT_TO_END_MIN, equal_nan=True),
        "같은 원시각에서 분 단위 계산")
expected_parts = changes_raw.loc[changes_raw.WO_NO.isin(cal.WO_NO)]
checked("교체 원행 보존", len(cal_parts) == len(expected_parts), f"{len(cal_parts)}행; 임의 중복 삭제·수량 합산 없음")
checked("부품·설비 마스터 연결", cal_parts.MASTER_LINK.eq("both").all() and len(equipment) == 1,
        "EQP_TAG·PART_CD 명시적 키")
checked("교체 표기·연결 일치", not cal.PART_FLAG_MISMATCH.any(), "PART_CHG와 부품교체 WO 연결 비교")
checked("모든 실제 작업 겹침 양수", overlaps.OVERLAP_MIN.gt(0).all(), f"{len(overlaps)}쌍; [착수,완료) 경계")
checked("전체 작업표 순서·누락·중복", table_covered == cal.WO_NO.tolist(), f"{page_count}쪽, {len(table_covered)}건")
expected_figures = page_count + len(timeline_files) + 1 + len(cases)
checked("그림 생성", len(saved_figures) == expected_figures
        and all((OUTPUT_DIR / name).stat().st_size > 0 for name in saved_figures), f"{len(saved_figures)}개")
checked("정지 길이 원문과 일치", joined.DOWN_MIN.equals(pd.to_numeric(original.DOWN_MIN, errors="coerce")),
        "정지 시각을 생성하지 않고 길이만 사용")
checked("분기별 통합 그림의 44건 누락·중복", timeline_covered == chronology.WO_NO.tolist(),
        "감지 시각순, 전체 작업을 분기별로 한 번씩 표시")
checked("그림 글자 경계", not layout_issues, str(layout_issues[:8]) if layout_issues else "그림 바깥·통합표 행/열 경계 검사")
display(pd.DataFrame(checks))

long_wait = cal.loc[cal.WAIT_MIN.idxmax()]
long_down = cal.loc[cal.DOWN_MIN.idxmax()]
run_summary = {
    "maintenance_count": len(cal), "type_counts": cal.MNT_TYPE.value_counts().to_dict(),
    "part_rows": len(cal_parts), "part_work_orders": int(cal_parts.WO_NO.nunique()),
    "zero_down_count": int(cal.DOWN_MIN.eq(0).sum()),
    "remark_empty_count": int(cal.REMARK_EMPTY.sum()),
    "remark_whitespace_count": int(cal.REMARK_WHITESPACE.sum()),
    "longest_wait_wo": long_wait.WO_NO, "longest_wait_min": float(long_wait.WAIT_MIN),
    "max_down_wo": long_down.WO_NO, "max_down_min": float(long_down.DOWN_MIN),
    "max_down_work_min": float(long_down.WORK_MIN),
    "overlap_pairs": len(overlaps),
    "overlap_details": [
        {"wo_a": row.WO_A, "wo_b": row.WO_B, "minutes": float(row.OVERLAP_MIN)}
        for row in overlaps.itertuples()
    ],
    "case_count": len(cases), "figure_count": len(saved_figures), "checks": len(checks),
    "figures": saved_figures,
    "work_observed_orders": int(cal.TEMP_WORK_ROWS.gt(0).sum()),
    "detection_only_orders": int((cal.TEMP_WORK_ROWS.eq(0) & cal.TEMP_DETECT_ROWS.gt(0)).sum()),
    "lot_count": len(lot_observations), "plan_orders": int(cal.PLAN_IDS.ne("").sum()),
    "layout_issues": layout_issues,
}
print("06_SUMMARY " + json.dumps(run_summary, ensure_ascii=False))


## 9. 결과를 읽을 때와 다음 분석 연결

- 대기시간은 감지~착수의 기록 차이다. CBM의 긴 대기는 점검 일정과 관련될 수 있으나 사유는 원문 이상으로 확정하지 않는다.
- DOWN_MIN=0은 유효한 원본값이다. 작업시간이 길어도 설비 정지가 있었다고 단정하지 않는다.
- TBM의 빈 FAIL_MODE는 그대로 두고, TBM에 값이 적힌 사례도 정비유형만으로 무효화하지 않는다.
- REMARK의 “재발”, “확인 중”, “원인 미상”은 자유서술이다. 반복 고장의 동일성·원인을 확정하는 라벨로 사용하지 않는다.
- 교체 부품·수량·사유는 WO로 연결한 근거다. 실제 교체 시각이나 교체 효과는 이 기록만으로 알 수 없다.
- 동시에 진행된 WO의 작업시간과 DOWN_MIN을 단순 합산해 설비 실제 정지시간으로 해석하지 않는다.
- 새 온도 대조 절은 감지 전·대기·작업·완료 후 관측을 분리하고 TC·OP·IQR과 같은 강종 대조를 확인한다. #76의 인접 측정간격 분석과 연결하되 관측이 없는 작업도 유지한다.

**재현:** 저장소 Python 환경에서 이 파일을 처음부터 끝까지 실행한다. 노트북 출력은 nbstripout으로 제거하고,
검토용 결과는 아래 그림으로 공유한다. 실제 실행 결과는 검증 완료 후 이 절 아래에 남긴다.


### 실행 확인 · 2026-10-15

- CAL01 정비 44건. BM 12건·CBM 10건·TBM 22건.
- 부품교체 8 WO·9행, 정지 0 7건.
- 실제 온도 측정 행이 작업 중 존재하는 WO 2건, 감지 시각에는 있지만 작업 중에는 없는 WO 1건.
- 관측 LOT 14개. CR1 정기수리 캘린더와 작업 구간이 겹치는 WO 11건.
- 기존 정비 원문·시간 시각화 18개. 새 온도 대조 그림 수와 계산 검증은 앞의 추가 분석 실행 결과에서 확인한다.

실행 방법: 프로젝트 Python 환경에서 코드 셀을 같은 프로세스 안에서 순서대로 실행했다. Jupyter 커널 자동 실행 검증은 하지 않았다.

### 저장 그림

- [01_전체작업표_순번01-11.png](../figures/06_maintenance_timeline_jinny/01_전체작업표/01_전체작업표_순번01-11.png)
- [02_전체작업표_순번12-22.png](../figures/06_maintenance_timeline_jinny/01_전체작업표/02_전체작업표_순번12-22.png)
- [03_전체작업표_순번23-33.png](../figures/06_maintenance_timeline_jinny/01_전체작업표/03_전체작업표_순번23-33.png)
- [04_전체작업표_순번34-44.png](../figures/06_maintenance_timeline_jinny/01_전체작업표/04_전체작업표_순번34-44.png)
- [01_CAL01_정비44건_전체.png](../figures/06_maintenance_timeline_jinny/02_통합타임라인/01_CAL01_정비44건_전체.png)
- [02_01-03월_정비기록.png](../figures/06_maintenance_timeline_jinny/02_통합타임라인/02_01-03월_정비기록.png)
- [03_04-06월_정비기록.png](../figures/06_maintenance_timeline_jinny/02_통합타임라인/03_04-06월_정비기록.png)
- [04_07-09월_정비기록.png](../figures/06_maintenance_timeline_jinny/02_통합타임라인/04_07-09월_정비기록.png)
- [05_10-12월_정비기록.png](../figures/06_maintenance_timeline_jinny/02_통합타임라인/05_10-12월_정비기록.png)
- [01_대기시간_작업시간_정지시간_비교.png](../figures/06_maintenance_timeline_jinny/03_시간비교/01_대기시간_작업시간_정지시간_비교.png)
- [01_최장대기_WO-2024-00627.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/01_최장대기_WO-2024-00627.png)
- [02_최대정지시간_WO-2024-01346.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/02_최대정지시간_WO-2024-01346.png)
- [03_정지0_최장작업_WO-2024-01467.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/03_정지0_최장작업_WO-2024-01467.png)
- [04_복수부품교체_WO-2024-00916.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/04_복수부품교체_WO-2024-00916.png)
- [05_11월09일_작업겹침_WO-2024-01823_WO-2024-01826.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/05_11월09일_작업겹침_WO-2024-01823_WO-2024-01826.png)
- [06_11월13일_작업겹침_WO-2024-01859_WO-2024-01854.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/06_11월13일_작업겹침_WO-2024-01859_WO-2024-01854.png)
- [07_07월08일_절연저하_교체_WO-2024-01094.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/07_07월08일_절연저하_교체_WO-2024-01094.png)
- [08_12월28일_마모량초과_교체_WO-2024-02105.png](../figures/06_maintenance_timeline_jinny/04_관심작업상세/08_12월28일_마모량초과_교체_WO-2024-02105.png)